# Marker gene deep analysis — selecting a minimal, orthogonal correlation panel

Metric group **L** (`mk_*`) scores every harmonization attempt with a Spearman correlation
computed over the marker panel and averaged into a handful of scalars. This notebook opens
those scalars up at the level of individual genes and gene sets, and works towards a
**minimal, orthogonal, conservative gene panel** to replace the full one.

Three problems motivate it, all verified on the 2026-08-26 tables:

| # | Problem | Evidence |
|---|---|---|
| 1 | The panel is **not measured uniformly** | No gene is present in all 2,407 in-scope attempts. Coverage is driven by the filter *strategy* (platform), not by the method |
| 2 | The panel is **redundant** | 63 signatures from overlapping publications share genes and biology, so the panel mean over-weights whichever biology has the most genes |
| 3 | Most of the panel **saturates** | 333 of 572 genes have a median ρ above 0.99 — any per-batch monotone transform scores ~1.0 by construction |

**Coverage is therefore defined on the unharmonized reference**: genes present in all 42
`01_raw__post0` matrices (14 strategies × 3 imputations). That is the harmonization-independent
answer, and it is this notebook's default gene set.

**Selection is driven by the spread of ρ across attempts, not by its height.** A gene that
scores 0.99 for every harmonizer carries no information about harmonizer quality. The reduced
panel is *not* asked to reproduce the full panel's ranking of attempts — that ranking is itself
dominated by the saturated genes — see §10 for what it is asked to do instead.

---

### Section map

| § | Contents | Inputs |
|---|---|---|
| 1 | Setup, paths, palettes | — |
| 2 | Load and assemble; analysis scope | metrics CSV, gene long table, marker annotation |
| 3 | Gene set structure: Venn, supervenn, Jaccard | marker annotation |
| 4 | Coverage analysis and the coverage gate | gene long table (+ coverage CSV) |
| 5 | Expression quality gate | `gene_qc_long` |
| 6 | Saturation: separating the permanently-high genes | gene long table |
| 7 | Distributions, quantiles, moments of ρ | gene long table |
| 8 | Master clustermap; raw vs harmonized scatterplots | gene long table; S3 |
| 9 | Gene–gene correlation structure | consensus matrices |
| 10 | Minimal orthogonal panel selection | consensus matrices |
| 11 | Variance, noise, coherence | `gene_qc_long`, consensus matrices |
| 12 | Recommendation and export | — |

Sections 5, 9, 10 and 11 need the tables produced by
`../harmonization-metrics-calculation/gene_panel_analysis/`. Until that run lands they print
a notice and skip; everything else works on the tables already in `metric_tables/`.

Generated by `create_marker_gene_deep_analysis_notebook.py`. Regenerate rather than
hand-edit. Plan: `marker_gene_deep_analysis_plan_260827.md`.

## §1 — Setup, paths and palettes

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import to_hex

warnings.filterwarnings("ignore")

# Publication defaults. 7.5 pt is the floor for every text element in these figures.
FONT_SIZE = 7.5
plt.rcParams.update({
    "font.size":            FONT_SIZE,
    "axes.labelsize":       FONT_SIZE,
    "axes.titlesize":       FONT_SIZE + 0.5,
    "xtick.labelsize":      FONT_SIZE,
    "ytick.labelsize":      FONT_SIZE,
    "legend.fontsize":      FONT_SIZE,
    "figure.titlesize":     FONT_SIZE + 1.5,
    "pdf.fonttype":         "truetype",
    "svg.fonttype":         "none",
    "figure.dpi":           200,
    "savefig.bbox":         "tight",
})
sns.set_style("ticks")

# ── Input paths. Every file this notebook reads is named here and nowhere else. ──
DATE_TAG   = "260826"     # metrics + Group L long tables (run_metrics_concat.py)
PANEL_TAG  = "260827"    # gene_panel_analysis outputs (run_gene_corr_concat.py)

METRIC_TABLES        = Path("metric_tables")
METRICS_CSV          = METRIC_TABLES / f"metrics_comprehensive_{DATE_TAG}.csv"
GENE_LONG_CSV        = METRIC_TABLES / f"marker_gene_correlations_long_{DATE_TAG}.csv"
COHORT_LONG_CSV      = METRIC_TABLES / f"marker_cohort_correlations_long_{DATE_TAG}.csv"
MARKER_ANN_CSV       = Path("../harmonization-metrics-calculation/marker_gene_annotation.csv")
COVERAGE_CSV         = METRIC_TABLES / f"marker_gene_coverage_by_attempt_{PANEL_TAG}.csv"
GENE_QC_CSV          = METRIC_TABLES / f"gene_qc_long_{PANEL_TAG}.csv.gz"
CONSENSUS_WITHIN_CSV = METRIC_TABLES / f"gene_gene_consensus_within_{PANEL_TAG}.csv.gz"
CONSENSUS_GLOBAL_CSV = METRIC_TABLES / f"gene_gene_consensus_global_{PANEL_TAG}.csv.gz"
CONSENSUS_SD_CSV     = METRIC_TABLES / f"gene_gene_consensus_sd_{PANEL_TAG}.csv.gz"
CONSENSUS_N_CSV      = METRIC_TABLES / f"gene_gene_consensus_n_{PANEL_TAG}.csv.gz"

# §8b downloads a handful of expression matrices straight from S3.
S3_BUCKET     = "$FL_S3_BUCKET"
S3_PREFIX     = "FL_batch_correction"
EXP_CACHE_DIR = Path("/tmp/fl_exp_cache")

FIG_DIR = Path("../figures_for_article/current_figures_tables_for_article_260819")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Analysis scope: the Shambhala cross-product is excluded from Article 1, but its default
# variant is kept and renamed, exactly as harmonization_metrics_analysis_v3.ipynb does.
SHAMBHALA_DEFAULT = "shambhala_P0std_Q0std"
SHAMBHALA_RENAMED = "20_shambhala"


def save_figure(fig, name, figures_dir=FIG_DIR):
    """Save one figure as PDF, SVG and PNG. PDF/SVG are the project defaults."""
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)
    for ext, kw in (("pdf", {}), ("svg", {}), ("png", {"dpi": 200})):
        fig.savefig(figures_dir / f"{name}.{ext}", **kw)
    print(f"saved: {figures_dir / name}.{{pdf,svg,png}}")


def load_optional_csv(path, **kwargs):
    """
    Read a table that may not exist yet.

    Sections 5, 9, 10 and 11 consume tables produced by gene_panel_analysis/. Returning
    an empty frame instead of raising keeps the notebook openable before that run lands.
    """
    path = Path(path)
    if not path.exists():
        print(f"NOT AVAILABLE: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, **kwargs)
    print(f"loaded {path}  ({df.shape[0]} x {df.shape[1]})")
    return df


print("Setup done.")

### §1.1 — Attempt palettes

Copied verbatim from `harmonization_metrics_analysis_v3.ipynb` (cells 4 and 34) so every
figure here is colour-identical to Figures 3–5 of the manuscript. Do not re-derive them:
`method_pal` in particular depends on the exact `tab20 + tab20b` order and the saturation
boost.

In [ ]:
# ── Harshness level per method (v3 cell 4) ───────────────────────────────────
HARSHNESS_PAL = {
    "low":    "#2ecc71",   # green
    "medium": "#f39c12",   # amber
    "high":   "#e74c3c",   # red
}

HARSHNESS_LEVEL_MAP = {
    "01_raw": "low", "02_median_scaling": "low", "03_limma": "low", "04_sva": "low",
    "05_combat": "medium", "06_combat_seq": "medium", "07_pycombat": "medium",
    "08_inmoose_combatseq": "medium", "09_ruv": "medium", "10_mnn": "medium",
    "11_harmony": "medium", "12_scanorama": "medium", "13_fsmvn": "medium",
    "14_qsmooth": "high", "15_fsqn_py": "high", "16_fsqn_r": "high",
    "17_quantile": "high", "18_rank": "high", "19_tdm": "high", "20_shambhala": "high",
    "21_harmonizr": "medium", "22_tmm": "low", "23_vst": "low", "24_peer_k10": "medium",
    "25_angel": "high", "26_xpn": "high", "27_dwd": "medium", "28_npn": "high",
    "29_combat_ref": "medium", "30_recombat": "medium", "31_ruv3prps": "medium",
    "32_deepmnn": "medium", "33_amdbnorm": "high", "34_arsyn": "medium",
    "35_dasc": "medium", "36_explobatch": "medium", "37_fabatch": "medium",
    "38_harman": "medium", "39_procrustes": "high",
}

# ── Strategy / imputation / post-removal palettes (v3 cell 34) ───────────────
ALL_STRATS = [
    "S0_no_removal", "A_confirmed_bad", "B_extended_bad", "C_rnaseq_only",
    "D_malignant_only", "E1_iterative_r1", "E2_iterative_r2", "E3_iterative_r3",
    "F_microarray_only", "G_affymetrix_only", "H_affymetrix_extended",
    "I_rare_batches_removed", "J_ff_only", "K_ffpe_only",
]

strat_pal = {
    "S0_no_removal":         (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad":       (1.0, 0.7058, 0.5098),
    "B_extended_bad":        (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only":         (1.0, 0.6235, 0.6078),
    "D_malignant_only":      (0.8156, 0.7333, 1.0),
    "E1_iterative_r1":       (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2":       (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3":       (0.8117, 0.8117, 0.8117),
    "F_microarray_only":     (1.0, 0.9960, 0.6392),
    "G_affymetrix_only":     (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only":             "#7ecbc4",   # soft teal — fresh frozen
    "K_ffpe_only":           "#f2c27f",   # warm amber — FFPE
}

imp_pal = {
    "strict":     "#2979ae",
    "knn":        "#982d22",
    "softimpute": "#713689",
}

post_rm_pal = {False: "#CCCCCC", True: "#333333"}


def _boost_saturation(rgb, delta: float = 0.15):
    h, s, v = mcolors.rgb_to_hsv(rgb[:3])
    return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))


def make_legend_patches(pal_dict):
    return [mpatches.Patch(color=to_hex(v), label=str(k)) for k, v in pal_dict.items()]


print(f"{len(strat_pal)} strategies, {len(imp_pal)} imputations, "
      f"{len(HARSHNESS_LEVEL_MAP)} methods in the harshness map.")

### §1.2 — Gene palettes

Two palettes this notebook adds.

**Cell type.** Reuses the `lymphoma_ontogeny_palette` of
`harmonization_metrics_analysis_v3.ipynb` (the `Diagnosis_cell_type_unified` palette, v3
cell 117) for every marker `cell_type` that maps onto a B-cell ontogeny stage, so a
centroblast is the same green here as in Figure 5. The nine microenvironment populations
have no counterpart there and are assigned by biology, following the project's
temperature-split rule: the malignant / GC B-cell axis is warm-to-green, adaptive lymphoid
cells are cold blues, myeloid is orange, stroma and vasculature are earth tones and pink.

**Gene set.** 63 `gene_group` values ordered by `source_article`, so signatures from the
same publication get neighbouring hues, built from `tab20 + tab20b + tab20c + Set2` with
the same saturation boost `method_pal` uses in v3.

In [ ]:
# ── Cell type palette ────────────────────────────────────────────────────────
# Six entries are taken straight from v3's lymphoma_ontogeny_palette; the nine TME
# populations have no ontogeny counterpart and are assigned by biology.
cell_type_pal = {
    # --- from v3 lymphoma_ontogeny_palette ---
    "centroblast (DZ)":     "#41ab5d",   # v3 "Centroblast"
    "centrocyte (LZ)":      "#a1d99b",   # v3 "Centrocyte"
    "germinal centre B":    "#006d2c",   # v3 "GC"
    "B cell":               "lawngreen", # v3 "B_cells"
    "plasma cell":          "grey",      # v3 "Plasma"
    "memory precursor":     "#00ced1",   # v3 "Memory"
    # --- new, by biology ---
    "CD4 T":                "#4A6FE3",   # adaptive lymphoid
    "CD8 T":                "#1F3B99",   # adaptive lymphoid
    "Treg":                 "#8FA9F0",   # adaptive lymphoid
    "NK":                   "#6A3D9A",   # innate lymphoid
    "macrophage":           "#FF7F00",   # myeloid
    "follicular dendritic": "#E6AB02",   # antigen-presenting stroma
    "stromal/fibroblast":   "#8C510A",   # stroma
    "endothelial":          "#E7298A",   # vasculature
    "none":                 "#EEEEEE",   # neutral
}

HOUSEKEEPING_PAL = {False: "#FFFFFF", True: "#000000"}

print(f"{len(cell_type_pal)} cell types in the palette.")

## §2 — Load and assemble

**Analysis scope.** The benchmark holds 3,835 attempts, of which 1,428 are Shambhala
cross-product variants excluded from Article 1. The default variant
`shambhala_P0std_Q0std` is kept and renamed `20_shambhala`, leaving **2,407 attempts**
(1,204 at post0). Every count in this notebook is on that scope unless stated otherwise.

In [ ]:
df_metrics = pd.read_csv(METRICS_CSV, low_memory=False)
df_metrics["run_id"] = (
    df_metrics["strat"] + "__" + df_metrics["imp"] + "__" + df_metrics["method"]
    + "__post" + df_metrics["post_rm"].map({False: "0", True: "1"})
)
n_all_attempts = len(df_metrics)

# Apply the analysis scope: drop the cross-product, keep and rename the default variant.
is_shambhala = df_metrics["method"].str.startswith("shambhala")
keep = (~is_shambhala) | (df_metrics["method"] == SHAMBHALA_DEFAULT)
df_metrics = df_metrics[keep & (df_metrics["status"] == "ok")].copy()
df_metrics.loc[df_metrics["method"] == SHAMBHALA_DEFAULT, "method"] = SHAMBHALA_RENAMED

# run_id keeps the original method spelling, so it still joins to the long tables.
df_metrics["harshness_level"] = df_metrics["method"].map(HARSHNESS_LEVEL_MAP).fillna("medium")
df_metrics = df_metrics.set_index("run_id")

SCOPE_RUN_IDS = set(df_metrics.index)

print(f"Attempts in the CSV        : {n_all_attempts}")
print(f"Attempts in analysis scope : {len(df_metrics)}")
print(f"  post0 only               : {(~df_metrics['post_rm']).sum()}")
print(f"  strategies               : {df_metrics['strat'].nunique()}")
print(f"  methods                  : {df_metrics['method'].nunique()}")
print(f"  imputations              : {df_metrics['imp'].nunique()}")

In [ ]:
rho_long = pd.read_csv(GENE_LONG_CSV)
rho_long = rho_long[rho_long["run_id"].isin(SCOPE_RUN_IDS)].copy()

_parts = rho_long["run_id"].str.split("__", expand=True)
rho_long["strat"], rho_long["imp"] = _parts[0], _parts[1]
rho_long["method"], rho_long["post"] = _parts[2], _parts[3]

# genes x attempts. About a third of the cells are NaN because coverage differs per
# strategy; that emptiness is itself the subject of section 4.
rho_wide = rho_long.pivot_table(index="gene", columns="run_id", values="rho")

print(f"Long table in scope : {len(rho_long):,} rows")
print(f"rho_wide            : {rho_wide.shape[0]} genes x {rho_wide.shape[1]} attempts")
print(f"  NaN fraction      : {np.isnan(rho_wide.values).mean():.1%}")
print(f"  rho range         : {rho_long['rho'].min():.3f} .. {rho_long['rho'].max():.3f}")
print(f"  rho median        : {rho_long['rho'].median():.3f}")

In [ ]:
ann_raw = pd.read_csv(MARKER_ANN_CSV)
ann_raw["is_housekeeping"] = ann_raw["is_housekeeping"].astype(bool)

# One row per (gene, gene_group) in the source table; collapse to one row per gene, since
# a gene may belong to several signatures.
GENE_GROUPS = sorted(ann_raw["gene_group"].unique())
_group_sizes = ann_raw.groupby("gene_group")["gene"].nunique().to_dict()


def _primary_group(groups):
    """The smallest signature a gene belongs to — its most specific membership."""
    return min(groups, key=lambda g: (_group_sizes[g], g))


ann = (
    ann_raw.groupby("gene")
    .agg(
        gene_groups=("gene_group", lambda s: sorted(set(s))),
        cell_types=("cell_type", lambda s: sorted(set(s) - {"none"})),
        pathways=("pathway", lambda s: sorted(set(s) - {"none"})),
        tme_subtypes=("tme_subtype", lambda s: sorted(set(s) - {"none"})),
        sources=("source_article", lambda s: sorted(set(s))),
        is_housekeeping=("is_housekeeping", "any"),
        n_groups=("gene_group", "nunique"),
    )
    .reset_index()
)
ann["primary_group"] = ann["gene_groups"].apply(_primary_group)
ann["primary_cell_type"] = ann["cell_types"].apply(lambda c: c[0] if c else "none")
ann["primary_pathway"] = ann["pathways"].apply(lambda p: p[0] if p else "none")
ann = ann.set_index("gene")

# Gene-set palette, ordered by source article so one publication's signatures are adjacent.
_group_order = (
    ann_raw.groupby("gene_group")["source_article"].first().sort_values().index.tolist()
)
_qual = (
    sns.color_palette("tab20", 20)
    + sns.color_palette("tab20b", 20)
    + sns.color_palette("tab20c", 20)
    + sns.color_palette("Set2", 8)
)
gene_group_pal = {
    g: to_hex(_boost_saturation(_qual[i % len(_qual)]))
    for i, g in enumerate(_group_order)
}

PATHWAYS = sorted(ann_raw["pathway"].unique())
TME_SUBTYPES = sorted(ann_raw["tme_subtype"].unique())
pathway_pal = dict(zip(PATHWAYS, [to_hex(c) for c in sns.color_palette("Dark2", len(PATHWAYS))]))
tme_subtype_pal = dict(
    zip(TME_SUBTYPES, [to_hex(c) for c in sns.color_palette("Set2", len(TME_SUBTYPES))])
)

print(f"Annotation      : {len(ann_raw)} (gene, gene_group) rows")
print(f"  unique genes  : {ann.index.nunique()}")
print(f"  gene groups   : {len(GENE_GROUPS)}")
print(f"  cell types    : {ann_raw['cell_type'].nunique()}")
print(f"  pathways      : {len(PATHWAYS)}")
print(f"  housekeeping  : {int(ann['is_housekeeping'].sum())}")
print(f"  genes never seen in any matrix: "
      f"{len(set(ann.index) - set(rho_wide.index))}")

In [ ]:
# Optional inputs from gene_panel_analysis/. Missing ones simply disable their sections.
coverage_tbl = load_optional_csv(COVERAGE_CSV)
gene_qc = load_optional_csv(GENE_QC_CSV)
consensus_within = load_optional_csv(CONSENSUS_WITHIN_CSV, index_col=0)
consensus_global = load_optional_csv(CONSENSUS_GLOBAL_CSV, index_col=0)
consensus_sd = load_optional_csv(CONSENSUS_SD_CSV, index_col=0)
consensus_n = load_optional_csv(CONSENSUS_N_CSV, index_col=0)

HAS_QC = not gene_qc.empty
HAS_CONSENSUS = not consensus_within.empty

print()
print(f"Group L tables      : available")
print(f"Coverage table      : {'available' if not coverage_tbl.empty else 'MISSING'}")
print(f"Expression QC       : {'available' if HAS_QC else 'MISSING — §5 and §11 will skip'}")
print(f"Consensus matrices  : {'available' if HAS_CONSENSUS else 'MISSING — §9 and §10 will skip'}")
if not HAS_QC or not HAS_CONSENSUS:
    print()
    print("To produce the missing tables, run in the metrics pod:")
    print("  cd harmonization-metrics-calculation/gene_panel_analysis")
    print("  python run_gene_corr_parallel.py --only-with-metrics --post-rm-filter post0 \\")
    print("      --n-workers 20 --memory-limit-gb 8.0 --timeout-s 1200")
    print(f"  python run_gene_corr_concat.py --out-dir <dir> --date-tag {PANEL_TAG} --by-strat")
    print(f"  python build_marker_gene_coverage.py --out-dir <dir> --date-tag {PANEL_TAG}")

In [ ]:
# Integrity checks — fail loudly rather than producing a quietly wrong figure.
assert set(rho_wide.columns) <= SCOPE_RUN_IDS, "rho_wide holds out-of-scope attempts"
missing_ann = set(rho_wide.index) - set(ann.index)
assert not missing_ann, f"genes in the long table with no annotation: {sorted(missing_ann)[:5]}"

print("Integrity checks passed.")
print(f"  {rho_wide.shape[0]} genes x {rho_wide.shape[1]} attempts, all annotated")

## §3 — Gene set structure: how much do the signatures overlap?

The panel is the union of 63 published signatures. Genes shared between signatures are
counted once in the panel but carry their biology into every set they belong to, so an
unweighted panel mean silently over-weights them. This section measures that overlap
before any filtering.

**Jaccard index** between two gene sets *A* and *B* is `|A ∩ B| / |A ∪ B|`: 0 = disjoint,
1 = identical. Blocks in the clustermap are families of signatures describing the same
biology from different publications — at most one member of a block should survive into
the final panel.

**supervenn** shows the actual set membership: each row is one signature, each column
a group of genes shared by exactly the same signatures. Wide single-signature columns mean
a signature is mostly unique; a tall narrow column is a gene shared by many signatures.

In [ ]:
# Membership matrix: genes x gene_groups
membership = (
    ann_raw.assign(v=1)
    .pivot_table(index="gene", columns="gene_group", values="v", fill_value=0)
    .astype(bool)
)
set_sizes = membership.sum(axis=0).sort_values(ascending=False)

print(f"Membership matrix: {membership.shape[0]} genes x {membership.shape[1]} sets")
print(f"Largest sets:\n{set_sizes.head(10)}")
print(f"\nGenes in more than one set: {(membership.sum(axis=1) > 1).sum()} "
      f"of {len(membership)}")

In [ ]:
# Fig l0_supervenn_top_sets — actual membership of the largest signatures.
N_TOP_SETS = 12
top_sets = set_sizes.head(N_TOP_SETS).index.tolist()

try:
    from supervenn import supervenn

    sets = [set(membership.index[membership[g]]) for g in top_sets]
    fig, ax = plt.subplots(figsize=(13, 6))
    supervenn(
        sets, top_sets, ax=ax, side_plots=True, chunks_ordering="size",
        widths_minmax_ratio=0.05, rotate_col_annotations=True, col_annotations_area_height=1.0,
    )
    ax.set_title(f"Gene membership of the {N_TOP_SETS} largest marker signatures")
    save_figure(fig, "l0_supervenn_top_sets")
    plt.show()
except ImportError:
    print("supervenn not installed — pip install supervenn")

In [ ]:
# Fig l0_venn_sources — overlap between the three published sources of the panel.
try:
    from matplotlib_venn import venn2, venn3

    src_short = {}
    for src in ann_raw["source_article"].unique():
        first = str(src).split()[0].strip(",")
        src_short.setdefault(first, []).append(src)
    src_genes = {
        short: set(ann_raw.loc[ann_raw["source_article"].isin(full), "gene"])
        for short, full in src_short.items()
    }
    top3 = sorted(src_genes, key=lambda k: -len(src_genes[k]))[:3]

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    venn3([src_genes[k] for k in top3], set_labels=top3, ax=axes[0])
    axes[0].set_title("Three largest source publications")

    gc_axis = set(ann_raw.loc[
        ann_raw["cell_type"].isin(
            ["centroblast (DZ)", "centrocyte (LZ)", "germinal centre B", "B cell"]
        ), "gene"])
    tme = set(ann_raw.loc[
        ann_raw["cell_type"].isin(
            ["CD4 T", "CD8 T", "Treg", "NK", "macrophage", "follicular dendritic",
             "endothelial", "stromal/fibroblast"]
        ), "gene"])
    venn2([gc_axis, tme], set_labels=["GC B-cell axis", "Microenvironment"], ax=axes[1])
    axes[1].set_title("Malignant axis vs microenvironment")

    fig.tight_layout()
    save_figure(fig, "l0_venn_sources")
    plt.show()
except ImportError:
    print("matplotlib_venn not installed — pip install matplotlib-venn")

In [ ]:
# Fig l0_jaccard_clustermap — pairwise Jaccard index between all 63 signatures.
M = membership.values.astype(float)
inter = M.T @ M
sizes = np.diag(inter)
union = sizes[:, None] + sizes[None, :] - inter
with np.errstate(invalid="ignore", divide="ignore"):
    jaccard = np.where(union > 0, inter / union, 0.0)
jaccard_df = pd.DataFrame(jaccard, index=membership.columns, columns=membership.columns)

row_colors = pd.DataFrame({
    "gene set": [gene_group_pal[g] for g in jaccard_df.index],
    "pathway": [
        pathway_pal.get(ann_raw.loc[ann_raw["gene_group"] == g, "pathway"].iloc[0], "#DDDDDD")
        for g in jaccard_df.index
    ],
}, index=jaccard_df.index)

g = sns.clustermap(
    jaccard_df, cmap="rocket_r", vmin=0, vmax=1, figsize=(12, 12),
    row_colors=row_colors, col_colors=row_colors,
    xticklabels=True, yticklabels=True,
    cbar_kws={"label": "Jaccard index"},
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=4.5, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=4.5)
g.figure.suptitle("Gene set overlap (Jaccard)", y=1.01)
save_figure(g.figure, "l0_jaccard_clustermap")
plt.show()

off_diag = jaccard[~np.eye(len(jaccard), dtype=bool)]
print(f"Mean off-diagonal Jaccard : {off_diag.mean():.3f}")
print(f"Set pairs with J > 0.5    : {(off_diag > 0.5).sum() // 2}")

In [ ]:
# Fig l0_gene_multiplicity — how many signatures each gene belongs to.
mult = membership.sum(axis=1).rename("n_sets").reset_index()
mult["primary_cell_type"] = mult["gene"].map(ann["primary_cell_type"])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
sns.histplot(data=mult, x="n_sets", bins=range(1, mult["n_sets"].max() + 2),
             color="#457B9D", ax=axes[0])
axes[0].set_xlabel("signatures the gene belongs to")
axes[0].set_ylabel("genes")
axes[0].set_title("Gene multiplicity")

top_mult = mult.nlargest(20, "n_sets").sort_values("n_sets")
sns.barplot(data=top_mult, y="gene", x="n_sets", hue="primary_cell_type",
            palette=cell_type_pal, dodge=False, ax=axes[1])
axes[1].set_title("20 most-shared genes")
axes[1].set_xlabel("signatures")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5)
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "l0_gene_multiplicity")
plt.show()

In [ ]:
# Table T0 — per-set summary and overlap.
T0 = pd.DataFrame({
    "gene_group": jaccard_df.index,
    "n_genes": [int(set_sizes[g]) for g in jaccard_df.index],
    "source_article": [
        ann_raw.loc[ann_raw["gene_group"] == g, "source_article"].iloc[0]
        for g in jaccard_df.index
    ],
    "cell_type": [
        ann_raw.loc[ann_raw["gene_group"] == g, "cell_type"].iloc[0]
        for g in jaccard_df.index
    ],
    "max_jaccard_other": [
        float(np.max(np.delete(jaccard_df.loc[g].values, jaccard_df.index.get_loc(g))))
        for g in jaccard_df.index
    ],
    "closest_other_set": [
        jaccard_df.columns[
            np.argsort(-np.where(jaccard_df.columns == g, -1, jaccard_df.loc[g].values))[0]
        ]
        for g in jaccard_df.index
    ],
})
T0 = T0.sort_values("max_jaccard_other", ascending=False)
T0.to_csv(FIG_DIR / "T0_gene_set_overlap.csv", index=False)
print(f"saved: {FIG_DIR / 'T0_gene_set_overlap.csv'}")
T0.head(12)

## §4 — Coverage and the coverage gate

A panel mean computed over a different gene set per attempt is not comparable across
attempts, so the first gate is coverage. Which coverage, though, is a real decision:

| `COVERAGE_SCOPE` | Definition | Genes |
|---|---|---|
| `"raw_all_pairs"` **(default)** | present in all 42 `01_raw__post0` matrices — 14 strategies × 3 imputations | 193 |
| `"reference_strategy"` | present in every attempt of `C_rnaseq_only` | 61 in scope (133 if the Shambhala variant is excluded) |
| `"global"` | present in ≥ `COVERAGE_MIN_FRAC` of all in-scope attempts | 150 at 95 %, **0** at 100 % |

Coverage measured over *harmonized* attempts is contaminated by the harmonizers: a method
that drops genes changes the denominator. The `01_raw__post0` definition removes that
circularity, which is why it is the default. `"global"` at 100 % returns nothing at all —
that is the single most important fact in this section.

In [ ]:
COVERAGE_SCOPE = "raw_all_pairs"      # "raw_all_pairs" | "reference_strategy" | "global"
COVERAGE_MIN_FRAC = 1.0               # used by "global"
REFERENCE_STRATEGY = "C_rnaseq_only"  # used by "reference_strategy"

# Coverage is derived from the long table itself: a (run_id, gene) row exists exactly when
# the gene was resolved in that attempt's matrix, whatever the rho value turned out to be.
raw_mask = (rho_long["method"] == "01_raw") & (rho_long["post"] == "post0")
raw_long = rho_long[raw_mask]
n_raw_pairs = raw_long["run_id"].nunique()
cov_raw = raw_long.groupby("gene")["run_id"].nunique()

n_scope = rho_long["run_id"].nunique()
cov_scope = rho_long.groupby("gene")["run_id"].nunique()

ref_long = rho_long[rho_long["strat"] == REFERENCE_STRATEGY]
n_ref = ref_long["run_id"].nunique()
cov_ref = ref_long.groupby("gene")["run_id"].nunique()

coverage = pd.DataFrame({
    "n_raw_pairs_present": cov_raw,
    "n_attempts_present_scope": cov_scope,
    "n_ref_strategy_present": cov_ref,
}).reindex(rho_wide.index).fillna(0).astype(int)
coverage["frac_raw_pairs"] = coverage["n_raw_pairs_present"] / n_raw_pairs
coverage["frac_attempts_scope"] = coverage["n_attempts_present_scope"] / n_scope
coverage["frac_ref_strategy"] = coverage["n_ref_strategy_present"] / max(n_ref, 1)
coverage["primary_cell_type"] = coverage.index.map(ann["primary_cell_type"])
coverage["primary_group"] = coverage.index.map(ann["primary_group"])
coverage["is_housekeeping"] = coverage.index.map(ann["is_housekeeping"])

if COVERAGE_SCOPE == "raw_all_pairs":
    covered_genes = coverage.index[coverage["frac_raw_pairs"] >= 1.0].tolist()
elif COVERAGE_SCOPE == "reference_strategy":
    covered_genes = coverage.index[coverage["frac_ref_strategy"] >= 1.0].tolist()
else:
    covered_genes = coverage.index[coverage["frac_attempts_scope"] >= COVERAGE_MIN_FRAC].tolist()

print(f"Raw reference matrices (01_raw__post0) : {n_raw_pairs}")
print(f"In-scope attempts                      : {n_scope}")
print(f"{REFERENCE_STRATEGY} attempts               : {n_ref}")
print()
print(f"Genes in all raw pairs                 : {(coverage['frac_raw_pairs'] >= 1.0).sum()}")
print(f"Genes in every {REFERENCE_STRATEGY} attempt  : {(coverage['frac_ref_strategy'] >= 1.0).sum()}")
print(f"Genes in every in-scope attempt        : {(coverage['frac_attempts_scope'] >= 1.0).sum()}")
print(f"Genes in >= 95% of in-scope attempts   : {(coverage['frac_attempts_scope'] >= 0.95).sum()}")
print()
print(f"COVERAGE_SCOPE = {COVERAGE_SCOPE!r} -> {len(covered_genes)} genes retained")

In [ ]:
# Fig l1_coverage_histogram — the two coverage definitions side by side.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
sns.histplot(coverage["frac_raw_pairs"], bins=30, color="#457B9D", ax=axes[0])
axes[0].set_title(f"Coverage over the {n_raw_pairs} raw reference matrices")
axes[0].set_xlabel("fraction of 01_raw__post0 matrices containing the gene")

sns.histplot(coverage["frac_attempts_scope"], bins=30, color="#E63946", ax=axes[1])
axes[1].set_title(f"Coverage over the {n_scope} in-scope attempts")
axes[1].set_xlabel("fraction of attempts containing the gene")

for ax in axes:
    for gate, style in ((0.95, ":"), (0.99, "--"), (1.0, "-")):
        ax.axvline(gate, color="k", ls=style, lw=0.8)
    ax.set_ylabel("genes")
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "l1_coverage_histogram")
plt.show()

In [ ]:
# Fig l1_coverage_by_strategy_heatmap — coverage is a property of the strategy.
cov_by_strat = (
    rho_long.groupby(["gene", "strat"])["run_id"].nunique().unstack(fill_value=0)
)
attempts_per_strat = rho_long.groupby("strat")["run_id"].nunique()
cov_by_strat = cov_by_strat.div(attempts_per_strat, axis=1).reindex(rho_wide.index).fillna(0)

row_colors = pd.DataFrame({
    "gene set": [gene_group_pal.get(ann.loc[g, "primary_group"], "#DDDDDD") for g in cov_by_strat.index],
    "cell type": [cell_type_pal.get(ann.loc[g, "primary_cell_type"], "#EEEEEE") for g in cov_by_strat.index],
}, index=cov_by_strat.index)
col_colors = pd.Series(
    [to_hex(strat_pal.get(s, "#AAAAAA")) for s in cov_by_strat.columns],
    index=cov_by_strat.columns, name="strategy",
)

g = sns.clustermap(
    cov_by_strat, cmap="mako_r", vmin=0, vmax=1, figsize=(7, 11),
    row_colors=row_colors, col_colors=col_colors,
    yticklabels=False, xticklabels=True, col_cluster=True,
    cbar_kws={"label": "fraction of the strategy's attempts"},
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=90)
g.ax_heatmap.set_ylabel(f"{len(cov_by_strat)} marker genes")
g.figure.suptitle("Marker gene coverage by filter strategy", y=1.0)
save_figure(g.figure, "l1_coverage_by_strategy_heatmap")
plt.show()

In [ ]:
# Fig l1_coverage_raw_vs_attempts — how much gene loss the harmonizers themselves cause.
fig, ax = plt.subplots(figsize=(4.6, 4.2))
sns.scatterplot(
    data=coverage.reset_index(), x="frac_raw_pairs", y="frac_attempts_scope",
    hue="primary_cell_type", palette=cell_type_pal, s=16, alpha=0.85,
    edgecolor="none", ax=ax,
)
ax.plot([0, 1], [0, 1], color="k", lw=0.8, ls="--")
ax.set_xlabel("coverage over raw reference matrices")
ax.set_ylabel("coverage over all in-scope attempts")
ax.set_title("Points below the diagonal are genes\nlost by harmonization, not by platform")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5, ncol=1)
sns.despine(fig=fig)
save_figure(fig, "l1_coverage_raw_vs_attempts")
plt.show()

In [ ]:
# Fig l1_coverage_vs_rho — is a poorly covered gene also a poorly behaved one?
gene_stats = pd.DataFrame({
    "median_rho": rho_wide.median(axis=1),
    "mean_rho": rho_wide.mean(axis=1),
    "sd_rho": rho_wide.std(axis=1),
    "p05_rho": rho_wide.quantile(0.05, axis=1),
    "p95_rho": rho_wide.quantile(0.95, axis=1),
    "n_attempts": rho_wide.notna().sum(axis=1),
})
gene_stats = gene_stats.join(coverage[["frac_raw_pairs", "frac_attempts_scope",
                                       "primary_cell_type", "primary_group",
                                       "is_housekeeping"]])
gene_stats["n_sets"] = gene_stats.index.map(ann["n_groups"])

fig, ax = plt.subplots(figsize=(5.0, 4.0))
sns.scatterplot(
    data=gene_stats.reset_index(), x="frac_attempts_scope", y="median_rho",
    size="sd_rho", hue="primary_cell_type", palette=cell_type_pal,
    sizes=(6, 70), alpha=0.8, edgecolor="none", ax=ax,
)
ax.set_xlabel("coverage over in-scope attempts")
ax.set_ylabel("median ρ across attempts")
ax.set_title("Coverage vs behaviour (point size = SD of ρ)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5)
sns.despine(fig=fig)
save_figure(fig, "l1_coverage_vs_rho")
plt.show()

In [ ]:
# Fig l1_coverage_gate_sweep — what a given threshold actually costs.
grid = np.linspace(0.5, 1.0, 51)
sweep = pd.DataFrame({
    "threshold": np.tile(grid, 3),
    "scope": np.repeat(["raw reference pairs", "in-scope attempts",
                        REFERENCE_STRATEGY], len(grid)),
    "n_genes": (
        [int((coverage["frac_raw_pairs"] >= t).sum()) for t in grid]
        + [int((coverage["frac_attempts_scope"] >= t).sum()) for t in grid]
        + [int((coverage["frac_ref_strategy"] >= t).sum()) for t in grid]
    ),
})

fig, ax = plt.subplots(figsize=(5.2, 3.4))
sns.lineplot(data=sweep, x="threshold", y="n_genes", hue="scope", lw=1.4, ax=ax)
ax.axvline(COVERAGE_MIN_FRAC, color="k", ls="--", lw=0.8)
ax.set_xlabel("minimum coverage fraction")
ax.set_ylabel("genes retained")
ax.set_title("Coverage gate sweep")
ax.legend(frameon=False, fontsize=6, title=None)
sns.despine(fig=fig)
save_figure(fig, "l1_coverage_gate_sweep")
plt.show()

T1 = sweep.pivot(index="threshold", columns="scope", values="n_genes").reset_index()
T1.to_csv(FIG_DIR / "T1_coverage_gate.csv", index=False)
print(f"saved: {FIG_DIR / 'T1_coverage_gate.csv'}")
T1[T1["threshold"].isin([0.5, 0.8, 0.9, 0.95, 0.99, 1.0])]

## §5 — Expression quality gate

A gene that is zero, or barely above zero, in most samples cannot carry a meaningful
within-cohort rank correlation: its ρ is noise dressed up as a measurement. This gate
removes those genes.

**Where the thresholds are evaluated.** On the `01_raw` rows only. All expression is
`log2(x + 1)`, so `frac_lt_1` counts samples below 1 on that scale (≈ linear expression
below 1). Most harmonizers shift and rescale the values, so the same threshold stops
meaning the same thing after harmonization — but the statistic is computed for every
attempt anyway, so the change each method causes can itself be plotted.

| Statistic | Meaning | Gate |
|---|---|---|
| `zero_frac` | fraction of samples at exactly 0 | ≤ `ZERO_FRAC_MAX` |
| `frac_lt_1` | fraction below 1 on the log2 scale | ≤ `LOW_FRAC_MAX` |
| `detection_frac` | fraction above 0 | ≥ `MIN_DETECTION_FRAC` |

Requires `gene_qc_long` from `gene_panel_analysis/`.

In [ ]:
ZERO_FRAC_MAX = 0.20        # a gene silent in >20% of samples cannot be ranked reliably
LOW_FRAC_MAX = 0.50         # more than half the samples in the noise band is too many
MIN_DETECTION_FRAC = 0.80   # the gene must be detected in most samples

if not HAS_QC:
    print("Expression QC table not available — §5 skipped.")
    qc_raw = pd.DataFrame()
    qc_gate_genes = list(rho_wide.index)
else:
    qc_raw = gene_qc[(gene_qc["method"] == "01_raw") & (~gene_qc["post_rm"])].copy()
    # Average the per-(strat, imp) reference values into one row per gene.
    qc_gene = qc_raw.groupby("gene").agg(
        zero_frac=("zero_frac", "mean"),
        frac_lt_1=("frac_lt_1", "mean"),
        detection_frac=("detection_frac", "mean"),
        mean_expr=("mean", "mean"),
        sd_expr=("sd", "mean"),
        cv=("cv", "mean"),
        icc_cohort=("icc_cohort", "mean"),
        genorm_m=("genorm_m", "mean"),
        var_between_cohort=("var_between_cohort", "mean"),
        var_within_cohort=("var_within_cohort", "mean"),
        n_raw_pairs=("run_id", "nunique"),
    )
    qc_gene["is_housekeeping"] = qc_gene.index.map(ann["is_housekeeping"])
    qc_gene["primary_cell_type"] = qc_gene.index.map(ann["primary_cell_type"])

    qc_pass = (
        (qc_gene["zero_frac"] <= ZERO_FRAC_MAX)
        & (qc_gene["frac_lt_1"] <= LOW_FRAC_MAX)
        & (qc_gene["detection_frac"] >= MIN_DETECTION_FRAC)
    )
    qc_gate_genes = qc_gene.index[qc_pass].tolist()
    print(f"QC evaluated on {qc_raw['run_id'].nunique()} raw reference matrices, "
          f"{len(qc_gene)} genes.")
    print(f"  pass zero_frac  <= {ZERO_FRAC_MAX}: {(qc_gene['zero_frac'] <= ZERO_FRAC_MAX).sum()}")
    print(f"  pass frac_lt_1  <= {LOW_FRAC_MAX}: {(qc_gene['frac_lt_1'] <= LOW_FRAC_MAX).sum()}")
    print(f"  pass detection  >= {MIN_DETECTION_FRAC}: {(qc_gene['detection_frac'] >= MIN_DETECTION_FRAC).sum()}")
    print(f"  pass all three                  : {len(qc_gate_genes)}")

In [ ]:
# Fig l2_zero_fraction_ecdf — where the thresholds actually cut.
if HAS_QC:
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
    sns.ecdfplot(data=qc_gene, x="zero_frac", hue="is_housekeeping",
                 palette={False: "#E63946", True: "#457B9D"}, ax=axes[0])
    axes[0].axvline(ZERO_FRAC_MAX, color="k", ls="--", lw=0.8)
    axes[0].set_title("Fraction of samples at exactly zero")

    sns.ecdfplot(data=qc_gene, x="frac_lt_1", hue="is_housekeeping",
                 palette={False: "#E63946", True: "#457B9D"}, ax=axes[1])
    axes[1].axvline(LOW_FRAC_MAX, color="k", ls="--", lw=0.8)
    axes[1].set_title("Fraction of samples below log2 = 1")

    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l2_zero_fraction_ecdf")
    plt.show()
else:
    print("skipped — no QC table")

In [ ]:
# Fig l2_qc_pairplot — the joint structure of the QC statistics.
if HAS_QC:
    cols = ["zero_frac", "frac_lt_1", "mean_expr", "sd_expr"]
    g = sns.pairplot(
        qc_gene.reset_index()[cols + ["is_housekeeping"]].dropna(),
        hue="is_housekeeping", palette={False: "#E63946", True: "#457B9D"},
        plot_kws={"s": 8, "alpha": 0.6, "edgecolor": "none"}, diag_kind="kde",
        height=1.5,
    )
    g.figure.suptitle("Expression QC statistics (01_raw)", y=1.02)
    save_figure(g.figure, "l2_qc_pairplot")
    plt.show()
else:
    print("skipped — no QC table")

In [ ]:
# Fig l2_zero_fraction_by_strategy — how much of the zero inflation is the platform.
if HAS_QC:
    strat_qc = gene_qc[(gene_qc["method"] == "01_raw") & (~gene_qc["post_rm"])]
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    order = sorted(strat_qc["strat"].unique())
    sns.boxplot(data=strat_qc, x="strat", y="zero_frac", order=order,
                palette={s: to_hex(strat_pal.get(s, "#AAAAAA")) for s in order},
                fliersize=0.5, linewidth=0.6, ax=ax)
    ax.axhline(ZERO_FRAC_MAX, color="k", ls="--", lw=0.8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_title("Zero inflation per filter strategy (unharmonized)")
    sns.despine(fig=fig)
    save_figure(fig, "l2_zero_fraction_by_strategy")
    plt.show()
else:
    print("skipped — no QC table")

In [ ]:
# Fig l2_zero_fraction_before_after — which methods create or destroy zeros.
if HAS_QC:
    per_method = gene_qc.groupby("method")["zero_frac"].median().sort_values()
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    sns.barplot(x=per_method.index, y=per_method.values,
                palette=[HARSHNESS_PAL[HARSHNESS_LEVEL_MAP.get(m, "medium")]
                         for m in per_method.index], ax=ax)
    ax.axhline(float(gene_qc.loc[gene_qc["method"] == "01_raw", "zero_frac"].median()),
               color="k", ls="--", lw=0.9, label="01_raw baseline")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_ylabel("median zero_frac")
    ax.set_title("Zero inflation per method (colour = harshness)")
    ax.legend(frameon=False, fontsize=6)
    sns.despine(fig=fig)
    save_figure(fig, "l2_zero_fraction_before_after")
    plt.show()
else:
    print("skipped — no QC table")

In [ ]:
# Fig l2_qc_gate_sweep + Table T2 — surviving gene counts over a 2-D threshold grid.
if HAS_QC:
    zero_grid = np.round(np.arange(0.05, 0.55, 0.05), 2)
    low_grid = np.round(np.arange(0.2, 0.95, 0.05), 2)
    counts = pd.DataFrame(
        [[int(((qc_gene["zero_frac"] <= z) & (qc_gene["frac_lt_1"] <= l)).sum())
          for l in low_grid] for z in zero_grid],
        index=zero_grid, columns=low_grid,
    )
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    sns.heatmap(counts, annot=True, fmt="d", annot_kws={"size": 5}, cmap="mako_r", ax=ax)
    ax.set_xlabel("LOW_FRAC_MAX")
    ax.set_ylabel("ZERO_FRAC_MAX")
    ax.set_title("Genes surviving the QC gate")
    save_figure(fig, "l2_qc_gate_sweep")
    plt.show()

    counts.to_csv(FIG_DIR / "T2_qc_gate.csv")
    print(f"saved: {FIG_DIR / 'T2_qc_gate.csv'}")
else:
    print("skipped — no QC table")

## §6 — Saturation: separating the permanently-high genes

Any harmonizer that applies a **per-batch monotone transform** of each gene leaves
within-cohort ranks untouched, so its Spearman ρ against the raw reference is 1.0 by
construction. A gene whose ρ is always ~1 therefore says nothing about harmonizer quality;
it only confirms that nothing exploded.

**The gate is on the median, not on a low quantile.** Verified on these tables: **no** gene
has a 5th-percentile ρ above 0.95 — every gene is broken by *some* attempt — while 333 of
572 genes have a median ρ above 0.99. Gating on `p05` would therefore remove nothing at all.

Removed genes are not discarded silently: the ordered quantile figure below shows the whole
panel at once, and the barplot lists exactly which biology leaves the working set.

In [ ]:
SATURATION_MEDIAN_MAX = 0.99   # genes whose median rho exceeds this carry no signal

sat = gene_stats.copy()
sat["saturated"] = sat["median_rho"] > SATURATION_MEDIAN_MAX

print(f"Genes with p05 rho    > 0.95 : {(sat['p05_rho'] > 0.95).sum()}")
print(f"Genes with p05 rho    > 0.90 : {(sat['p05_rho'] > 0.90).sum()}")
print(f"Genes with median rho > 0.99 : {(sat['median_rho'] > 0.99).sum()}")
print(f"Genes with median rho > 0.95 : {(sat['median_rho'] > 0.95).sum()}")
print()
print(f"SATURATION_MEDIAN_MAX = {SATURATION_MEDIAN_MAX} removes {int(sat['saturated'].sum())} "
      f"of {len(sat)} genes")

ref_genes = [g for g in covered_genes if g in sat.index]
print(f"Within the {len(ref_genes)}-gene coverage set: "
      f"{int(sat.loc[ref_genes, 'saturated'].sum())} saturated, "
      f"{int((~sat.loc[ref_genes, 'saturated']).sum())} retained")

In [ ]:
# Fig l3_gene_quantile_ordered — the whole panel at a glance, ordered by median rho.
ordered = sat.sort_values("median_rho")
x = np.arange(len(ordered))

fig, ax = plt.subplots(figsize=(8.0, 3.4))
ax.fill_between(x, ordered["p05_rho"], ordered["p95_rho"], color="#8D99AE", alpha=0.35,
                lw=0, label="p05–p95 across attempts")
ax.fill_between(x, ordered["median_rho"] - ordered["sd_rho"],
                ordered["median_rho"] + ordered["sd_rho"], color="#457B9D", alpha=0.35,
                lw=0, label="median ± SD")
sns.lineplot(x=x, y=ordered["median_rho"].values, color="#1D3557", lw=1.2,
             label="median", ax=ax)
ax.axhline(SATURATION_MEDIAN_MAX, color="#E63946", ls="--", lw=1.0,
           label=f"saturation gate ({SATURATION_MEDIAN_MAX})")
ax.set_xlabel("marker genes, ordered by median ρ")
ax.set_ylabel("Spearman ρ vs the unharmonized reference")
ax.set_title("Per-gene ρ across harmonization attempts")
ax.legend(frameon=False, fontsize=6, loc="lower right")
sns.despine(fig=fig)
save_figure(fig, "l3_gene_quantile_ordered")
plt.show()

In [ ]:
# Fig l3_saturated_genes_barplot — which genes leave the working set, and whose biology.
removed = sat[sat["saturated"]].sort_values("median_rho", ascending=False)
show = removed.head(60).reset_index()

fig, ax = plt.subplots(figsize=(9.0, 3.6))
sns.barplot(data=show, x="gene", y="median_rho", hue="primary_cell_type",
            palette=cell_type_pal, dodge=False, ax=ax)
ax.set_ylim(SATURATION_MEDIAN_MAX - 0.005, 1.001)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=4.5)
ax.set_ylabel("median ρ")
ax.set_title(f"Saturated genes removed by the gate (top 60 of {len(removed)})")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False, fontsize=5)
sns.despine(fig=fig)
save_figure(fig, "l3_saturated_genes_barplot")
plt.show()

In [ ]:
# Fig l3_retained_vs_removed_composition — does the gate delete an entire cell type?
comp = sat.reset_index()
comp["group"] = np.where(comp["saturated"], "removed (saturated)", "retained")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
sns.countplot(data=comp, y="primary_cell_type", hue="group",
              palette={"retained": "#457B9D", "removed (saturated)": "#E63946"},
              order=comp["primary_cell_type"].value_counts().index, ax=axes[0])
axes[0].set_title("Cell types")
axes[0].legend(frameon=False, fontsize=6)

top_groups = comp["primary_group"].value_counts().head(20).index
sns.countplot(data=comp[comp["primary_group"].isin(top_groups)], y="primary_group",
              hue="group",
              palette={"retained": "#457B9D", "removed (saturated)": "#E63946"},
              order=top_groups, ax=axes[1])
axes[1].set_title("Gene sets (20 largest)")
axes[1].legend(frameon=False, fontsize=6)
axes[1].tick_params(labelsize=5)
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "l3_retained_vs_removed_composition")
plt.show()

T3 = sat.reset_index()[
    ["gene", "median_rho", "mean_rho", "sd_rho", "p05_rho", "p95_rho", "n_attempts",
     "frac_raw_pairs", "frac_attempts_scope", "primary_cell_type", "primary_group",
     "is_housekeeping", "saturated"]
].sort_values("median_rho")
T3.to_csv(FIG_DIR / "T3_saturation_split.csv", index=False)
print(f"saved: {FIG_DIR / 'T3_saturation_split.csv'}")

## §7 — Distributions, quantiles and moments of ρ

The scalar `mk_rho_mean_all_genes` is one number per attempt taken over this whole
distribution. Here it is opened up by gene, by gene set, and by cell type.

The plot to read most carefully is **`l4_rho_sd_vs_median`**: the x axis is how *high* a
gene scores, the y axis is how much it *varies* between harmonizers. Selection needs the
right-hand column of that plot to be empty and the upper band to be populated — a gene is
useful when different harmonizers give it visibly different values.

In [ ]:
# Fig l4_rho_distribution_overall
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.0))
sns.histplot(rho_long["rho"].dropna(), bins=100, color="#457B9D", ax=axes[0])
axes[0].axvline(SATURATION_MEDIAN_MAX, color="#E63946", ls="--", lw=0.9)
axes[0].set_title(f"All {len(rho_long):,} (attempt, gene) correlations")
axes[0].set_xlabel("Spearman ρ")

sns.ecdfplot(rho_long["rho"].dropna(), color="#1D3557", ax=axes[1])
axes[1].axvline(SATURATION_MEDIAN_MAX, color="#E63946", ls="--", lw=0.9)
axes[1].set_title("ECDF — the shoulder near 1.0 is the saturation")
axes[1].set_xlabel("Spearman ρ")
sns.despine(fig=fig)
fig.tight_layout()
save_figure(fig, "l4_rho_distribution_overall")
plt.show()

print(rho_long["rho"].describe())

In [ ]:
# Fig l4_rho_by_gene_set — per-signature distributions, ordered by median.
rho_g = rho_long.merge(
    ann[["primary_group", "primary_cell_type", "is_housekeeping"]],
    left_on="gene", right_index=True, how="left",
)
order = rho_g.groupby("primary_group")["rho"].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(9.5, 4.2))
sns.violinplot(data=rho_g, x="primary_group", y="rho", order=order,
               palette=gene_group_pal, cut=0, linewidth=0.4, inner="quartile", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=4.5)
ax.set_xlabel("")
ax.set_ylabel("Spearman ρ")
ax.set_title("ρ distribution per gene set (primary membership)")
sns.despine(fig=fig)
save_figure(fig, "l4_rho_by_gene_set")
plt.show()

In [ ]:
# Fig l4_rho_by_cell_type — housekeeping shown as its own reference band.
rho_g["band"] = np.where(rho_g["is_housekeeping"], "housekeeping", rho_g["primary_cell_type"])
order = rho_g.groupby("band")["rho"].median().sort_values().index.tolist()
pal = {**cell_type_pal, "housekeeping": "#6C757D"}

fig, ax = plt.subplots(figsize=(7.5, 3.4))
sns.boxplot(data=rho_g, x="band", y="rho", order=order, palette=pal,
            fliersize=0.4, linewidth=0.6, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_xlabel("")
ax.set_ylabel("Spearman ρ")
ax.set_title("ρ by cell type, with the housekeeping controls")
sns.despine(fig=fig)
save_figure(fig, "l4_rho_by_cell_type")
plt.show()

In [ ]:
# Fig l4_rho_quantile_heatmap — every moment of every gene, z-scored per column.
qcols = ["mean_rho", "median_rho", "p05_rho", "p95_rho", "sd_rho"]
qmat = gene_stats[qcols].copy()
qmat["IQR"] = rho_wide.quantile(0.75, axis=1) - rho_wide.quantile(0.25, axis=1)
qmat["min"] = rho_wide.min(axis=1)
qmat = qmat.dropna()

# A column with no variance cannot be z-scored, and one silently NaNs the whole
# clustering. p95 is exactly 1.0 for every gene — every gene reaches ρ = 1 in its best
# 5% of attempts — which is the saturation of §6 seen from another angle.
constant_cols = qmat.columns[qmat.std() == 0].tolist()
if constant_cols:
    print(f"Dropped constant columns (identical for every gene): {constant_cols}")
    for col in constant_cols:
        print(f"  {col} = {qmat[col].iloc[0]:.4f} for all {len(qmat)} genes")
    qmat = qmat.drop(columns=constant_cols)

row_colors = pd.DataFrame({
    "gene set": [gene_group_pal.get(ann.loc[g, "primary_group"], "#DDDDDD") for g in qmat.index],
    "cell type": [cell_type_pal.get(ann.loc[g, "primary_cell_type"], "#EEEEEE") for g in qmat.index],
}, index=qmat.index)

g = sns.clustermap(
    qmat, z_score=1, cmap="vlag", center=0, figsize=(5.0, 10.0),
    row_colors=row_colors, col_cluster=False, yticklabels=False,
    cbar_kws={"label": "z-score within column"},
)
g.ax_heatmap.set_ylabel(f"{len(qmat)} marker genes")
g.figure.suptitle("Per-gene ρ moments", y=1.0)
save_figure(g.figure, "l4_rho_quantile_heatmap")
plt.show()

In [ ]:
# Fig l4_rho_sd_vs_median — the discriminative-power plot. This drives the selection.
fig, ax = plt.subplots(figsize=(5.4, 4.2))
plot_df = gene_stats.reset_index()
sns.scatterplot(data=plot_df, x="median_rho", y="sd_rho", hue="primary_cell_type",
                palette=cell_type_pal, s=18, alpha=0.85, edgecolor="none", ax=ax)
ax.axvline(SATURATION_MEDIAN_MAX, color="#E63946", ls="--", lw=0.9)

for _, r in plot_df.nlargest(25, "sd_rho").iterrows():
    ax.annotate(r["gene"], (r["median_rho"], r["sd_rho"]), fontsize=4,
                xytext=(1.5, 1.5), textcoords="offset points")

ax.set_xlabel("median ρ  (how high the gene scores)")
ax.set_ylabel("SD of ρ across attempts  (how much it discriminates)")
ax.set_title("Discriminative power: the y axis is what selection maximises")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5)
sns.despine(fig=fig)
save_figure(fig, "l4_rho_sd_vs_median")
plt.show()

In [ ]:
# Fig l4_rho_by_attempt_facets — the same genes seen through strategy and imputation.
g = sns.catplot(
    data=rho_g, x="imp", y="rho", col="strat", kind="box", col_wrap=5,
    palette=imp_pal, height=1.9, aspect=0.85, fliersize=0.3, linewidth=0.5,
    order=["strict", "knn", "softimpute"],
)
g.set_titles("{col_name}", size=6)
g.set_axis_labels("", "ρ")
g.figure.suptitle("ρ by imputation within each filter strategy", y=1.01)
save_figure(g.figure, "l4_rho_by_attempt_facets")
plt.show()

T4 = gene_stats.reset_index()
T4["gene_groups"] = T4["gene"].map(lambda g_: ";".join(ann.loc[g_, "gene_groups"]))
T4["cell_types"] = T4["gene"].map(lambda g_: ";".join(ann.loc[g_, "cell_types"]))
T4.to_csv(FIG_DIR / "T4_per_gene_rho_summary.csv", index=False)
print(f"saved: {FIG_DIR / 'T4_per_gene_rho_summary.csv'}")

## §8 — Master clustermap, and touching the data

### §8a — Genes × attempts

The headline figure: every gene against every attempt, with all five attempt palettes as
column tracks and five gene tracks as rows. Column blocks are groups of harmonizers that
treat the panel alike; row blocks are groups of genes that respond alike to harmonization.

In [ ]:
# Working gene set after the gates applied so far.
gate_genes = [g for g in covered_genes if g in rho_wide.index]
if HAS_QC:
    gate_genes = [g for g in gate_genes if g in set(qc_gate_genes)]
working_genes = [g for g in gate_genes if not sat.loc[g, "saturated"]]

print(f"After coverage ({COVERAGE_SCOPE})           : {len(covered_genes)}")
print(f"After the expression QC gate               : {len(gate_genes)}")
print(f"After the saturation gate                  : {len(working_genes)}")

In [ ]:
def attempt_col_colors(run_ids):
    """Five column tracks, all from the v3 palettes, for a list of run_ids."""
    meta = df_metrics.loc[list(run_ids)]
    return pd.DataFrame({
        "strategy":   [to_hex(strat_pal.get(s, "#AAAAAA")) for s in meta["strat"]],
        "imputation": [imp_pal.get(i, "#AAAAAA") for i in meta["imp"]],
        "post_rm":    [post_rm_pal[bool(p)] for p in meta["post_rm"]],
        "harshness":  [HARSHNESS_PAL[h] for h in meta["harshness_level"]],
    }, index=list(run_ids))


def gene_row_colors(genes):
    """Five row tracks: gene set, cell type, pathway, housekeeping, raw coverage."""
    grey = sns.color_palette("Greys", 11)
    return pd.DataFrame({
        "gene set":   [gene_group_pal.get(ann.loc[g, "primary_group"], "#DDDDDD") for g in genes],
        "cell type":  [cell_type_pal.get(ann.loc[g, "primary_cell_type"], "#EEEEEE") for g in genes],
        "pathway":    [pathway_pal.get(ann.loc[g, "primary_pathway"], "#DDDDDD") for g in genes],
        "housekeep":  [HOUSEKEEPING_PAL[bool(ann.loc[g, "is_housekeeping"])] for g in genes],
        "coverage":   [to_hex(grey[int(round(coverage.loc[g, "frac_raw_pairs"] * 10))]) for g in genes],
    }, index=list(genes))


# Fig l5_clustermap_genes_by_attempts — the full matrix.
mat = rho_wide.loc[gate_genes].dropna(axis=1, how="all")
mat = mat.fillna(mat.median(axis=1).median())   # clustering cannot consume NaN

g = sns.clustermap(
    mat, cmap="RdYlBu_r", vmin=0, vmax=1, figsize=(13, 11),
    col_colors=attempt_col_colors(mat.columns), row_colors=gene_row_colors(mat.index),
    xticklabels=False, yticklabels=False, method="average",
    cbar_kws={"label": "Spearman ρ"},
)
g.ax_heatmap.set_xlabel(f"{mat.shape[1]} harmonization attempts")
g.ax_heatmap.set_ylabel(f"{mat.shape[0]} marker genes")
g.figure.suptitle("Marker gene correlation preservation across the benchmark", y=1.0)
save_figure(g.figure, "l5_clustermap_genes_by_attempts")
plt.show()

In [ ]:
# Fig l5_clustermap_genes_by_method_median — legible in print: one column per method.
by_method = rho_wide.loc[gate_genes].T.join(df_metrics[["method"]]).groupby("method").median().T
by_method = by_method.dropna(how="all").fillna(by_method.median(axis=1).median())

method_colors = pd.DataFrame({
    "harshness": [HARSHNESS_PAL[HARSHNESS_LEVEL_MAP.get(m, "medium")] for m in by_method.columns],
}, index=by_method.columns)

g = sns.clustermap(
    by_method, cmap="RdYlBu_r", vmin=0, vmax=1, figsize=(8.5, 10),
    col_colors=method_colors, row_colors=gene_row_colors(by_method.index),
    xticklabels=True, yticklabels=False, method="average",
    cbar_kws={"label": "median ρ"},
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=5.5, rotation=90)
g.ax_heatmap.set_ylabel(f"{by_method.shape[0]} marker genes")
g.figure.suptitle("Per-method median ρ", y=1.0)
save_figure(g.figure, "l5_clustermap_genes_by_method_median")
plt.show()

In [ ]:
# Fig l5_clustermap_reference_set — only the raw-covered genes, the article's gene set.
ref_mat = rho_wide.loc[[g_ for g_ in covered_genes if g_ in rho_wide.index]]
ref_mat = ref_mat.dropna(axis=1, how="all")
ref_mat = ref_mat.fillna(ref_mat.median(axis=1).median())

g = sns.clustermap(
    ref_mat, cmap="RdYlBu_r", vmin=0, vmax=1, figsize=(12, 8),
    col_colors=attempt_col_colors(ref_mat.columns), row_colors=gene_row_colors(ref_mat.index),
    xticklabels=False, yticklabels=False, method="average",
    cbar_kws={"label": "Spearman ρ"},
)
g.ax_heatmap.set_ylabel(f"{ref_mat.shape[0]} genes covered in every raw reference matrix")
g.figure.suptitle("Coverage-complete gene set", y=1.0)
save_figure(g.figure, "l5_clustermap_reference_set")
plt.show()

In [ ]:
# Legend panels — 63 gene sets will not fit beside a heatmap, so they get their own figure.
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (title, pal, ncol) in zip(axes, [
    ("Cell type", cell_type_pal, 1),
    ("Strategy / imputation / harshness",
     {**{k: to_hex(v) for k, v in strat_pal.items()}, **imp_pal, **HARSHNESS_PAL}, 1),
    ("Pathway", pathway_pal, 1),
]):
    ax.axis("off")
    ax.legend(handles=make_legend_patches(pal), loc="upper left", frameon=False,
              fontsize=6, ncol=ncol, title=title)
save_figure(fig, "l5_clustermap_legends_main")
plt.show()

fig, ax = plt.subplots(figsize=(9, 8))
ax.axis("off")
ax.legend(handles=make_legend_patches(gene_group_pal), loc="upper left", frameon=False,
          fontsize=5, ncol=3, title="Gene set")
save_figure(fig, "l5_clustermap_legends_gene_sets")
plt.show()

### §8b — Raw vs harmonized scatterplots

Group L is a Spearman correlation between two vectors: a gene's expression across one
cohort's samples **before** harmonization and **after**. Everything above is a summary of
those vectors. This section plots them directly.

The figure that explains the metric best is `l6_scatter_all_cohorts_one_gene`: a gene can
look wildly scattered when all cohorts are pooled and still score ρ ≈ 1, because the metric
is computed *inside* each cohort.

Downloads about 2.5 GB from S3 into `/tmp/fl_exp_cache` on first run. Set
`RUN_SCATTER_SECTION = False` to skip.

In [ ]:
RUN_SCATTER_SECTION = True

SCATTER_ATTEMPTS = [
    "C_rnaseq_only__softimpute__04_sva__post0",
    "C_rnaseq_only__softimpute__10_mnn__post0",
    "C_rnaseq_only__softimpute__16_fsqn_r__post0",
    "G_affymetrix_only__softimpute__16_fsqn_r__post0",
]

import gzip
import io


def load_exp(strat, imp, method, post_rm):
    """Download one harmonized expression matrix from S3, cached on local disk."""
    import boto3

    pm = "1" if post_rm else "0"
    name = f"{strat}__{imp}__{method}__post{pm}.tsv.gz"
    EXP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached = EXP_CACHE_DIR / name
    if not cached.exists():
        body = boto3.client("s3").get_object(
            Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/exp/{name}"
        )["Body"].read()
        cached.write_bytes(body)
    return pd.read_csv(cached, sep="\t", index_col=0, low_memory=False)


def load_ann(strat, imp):
    """Download the prepared annotation aligned to those expression matrices."""
    import boto3

    name = f"{strat}__{imp}__ann.tsv.gz"
    EXP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached = EXP_CACHE_DIR / name
    if not cached.exists():
        body = boto3.client("s3").get_object(
            Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/prepared/{name}"
        )["Body"].read()
        cached.write_bytes(body)
    return pd.read_csv(cached, sep="\t", index_col=0, low_memory=False)


def paired_frame(run_id, genes):
    """
    Long frame of (sample, cohort, gene, raw, harmonized) for one attempt.

    The reference is always 01_raw__post0 intersected down to the attempt's own sample
    index — the same rule metric group L uses, so the numbers plotted here are the
    numbers the metric saw.
    """
    strat, imp, method, pm = run_id.split("__")
    post_rm = pm == "post1"
    harm = load_exp(strat, imp, method, post_rm)
    raw = load_exp(strat, imp, "01_raw", False)
    ann_s = load_ann(strat, imp)

    samples = harm.index.intersection(raw.index)
    genes = [g for g in genes if g in harm.columns and g in raw.columns]
    cohort_col = "COHORT_LABEL" if "COHORT_LABEL" in ann_s.columns else "RNA_BATCH"

    out = []
    for gene in genes:
        out.append(pd.DataFrame({
            "sample": samples,
            "cohort": ann_s.reindex(samples)[cohort_col].astype(str).values,
            "gene": gene,
            "raw": raw.loc[samples, gene].values,
            "harmonized": harm.loc[samples, gene].values,
        }))
    return pd.concat(out, ignore_index=True)


print("Loaders defined." if RUN_SCATTER_SECTION else "Scatter section disabled.")

In [ ]:
# Pick 9 genes spanning the whole range of the metric: 3 high, 3 mid, 3 low.
if RUN_SCATTER_SECTION:
    ranked = gene_stats.loc[[g for g in gate_genes if g in gene_stats.index]]
    ranked = ranked.sort_values("median_rho")
    SCATTER_GENES = (
        ranked.head(3).index.tolist()
        + ranked.iloc[len(ranked) // 2 - 1: len(ranked) // 2 + 2].index.tolist()
        + ranked.tail(3).index.tolist()
    )
    print("Genes chosen (low → high median ρ):")
    print(ranked.loc[SCATTER_GENES, ["median_rho", "sd_rho"]])
else:
    SCATTER_GENES = []

In [ ]:
# Fig l6_scatter_gene_cohort_grid — genes x cohorts, raw on x, harmonized on y.
if RUN_SCATTER_SECTION:
    from scipy.stats import spearmanr

    run_id = SCATTER_ATTEMPTS[0]
    pf = paired_frame(run_id, SCATTER_GENES)
    big_cohorts = pf["cohort"].value_counts().head(4).index.tolist()
    sub = pf[pf["cohort"].isin(big_cohorts)]

    g = sns.relplot(
        data=sub, x="raw", y="harmonized", col="cohort", row="gene",
        kind="scatter", s=5, alpha=0.5, color="#457B9D",
        height=1.5, aspect=1.0, facet_kws={"sharex": False, "sharey": False},
    )
    for (gene, cohort), ax in g.axes_dict.items():
        d = sub[(sub["gene"] == gene) & (sub["cohort"] == cohort)]
        if len(d) > 2:
            rho = spearmanr(d["raw"], d["harmonized"]).statistic
            ax.set_title(f"{gene} · {cohort[:14]}\nρ = {rho:.3f}", fontsize=5)
        lo = min(d["raw"].min(), d["harmonized"].min())
        hi = max(d["raw"].max(), d["harmonized"].max())
        ax.plot([lo, hi], [lo, hi], color="k", lw=0.6, ls="--")
    g.figure.suptitle(f"Raw vs harmonized — {run_id}", y=1.01)
    save_figure(g.figure, "l6_scatter_gene_cohort_grid")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l6_scatter_all_cohorts_one_gene — why pooled scatter and within-cohort ρ differ.
if RUN_SCATTER_SECTION:
    from scipy.stats import spearmanr

    gene = SCATTER_GENES[-1]
    pf = paired_frame(SCATTER_ATTEMPTS[0], [gene])
    top_cohorts = pf["cohort"].value_counts().head(10).index
    sub = pf[pf["cohort"].isin(top_cohorts)]
    cohort_palette = dict(zip(sorted(top_cohorts),
                              sns.color_palette("husl", len(top_cohorts))))

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    sns.scatterplot(data=sub, x="raw", y="harmonized", hue="cohort",
                    palette=cohort_palette, s=8, alpha=0.7, edgecolor="none", ax=axes[0])
    axes[0].set_title(f"{gene}: all cohorts pooled\n"
                      f"pooled ρ = {spearmanr(sub['raw'], sub['harmonized']).statistic:.3f}")
    axes[0].legend().remove()

    for cohort, d in sub.groupby("cohort"):
        sns.regplot(data=d, x="raw", y="harmonized", scatter=False, ci=None,
                    color=cohort_palette[cohort], line_kws={"lw": 1.0}, ax=axes[1])
    within = [spearmanr(d["raw"], d["harmonized"]).statistic
              for _, d in sub.groupby("cohort") if len(d) > 2]
    axes[1].set_title(f"{gene}: per-cohort regression lines\n"
                      f"mean within-cohort ρ = {np.nanmean(within):.3f}")
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l6_scatter_all_cohorts_one_gene")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l6_scatter_method_panel — the same gene and cohort through four harmonizers.
if RUN_SCATTER_SECTION:
    from scipy.stats import spearmanr

    gene = SCATTER_GENES[0]     # the worst-behaved gene of the working set
    fig, axes = plt.subplots(1, len(SCATTER_ATTEMPTS), figsize=(3.0 * len(SCATTER_ATTEMPTS), 3.0))
    for ax, run_id in zip(np.atleast_1d(axes), SCATTER_ATTEMPTS):
        try:
            pf = paired_frame(run_id, [gene])
        except Exception as exc:
            ax.set_title(f"{run_id}\nunavailable", fontsize=5)
            ax.axis("off")
            continue
        cohort = pf["cohort"].value_counts().index[0]
        d = pf[pf["cohort"] == cohort]
        sns.scatterplot(data=d, x="raw", y="harmonized", s=8, alpha=0.6,
                        color="#E63946", edgecolor="none", ax=ax)
        lo, hi = d["raw"].min(), d["raw"].max()
        ax.plot([lo, hi], [lo, hi], color="k", lw=0.6, ls="--")
        rho = spearmanr(d["raw"], d["harmonized"]).statistic if len(d) > 2 else np.nan
        ax.set_title(f"{run_id.split('__')[2]}\n{gene} · {cohort[:14]} · ρ={rho:.3f}", fontsize=5)
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l6_scatter_method_panel")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l6_density_before_after — cold = raw, warm = harmonized, per the palette rule.
if RUN_SCATTER_SECTION:
    gene = SCATTER_GENES[-1]
    pf = paired_frame(SCATTER_ATTEMPTS[1], [gene])
    top_cohorts = pf["cohort"].value_counts().head(6).index
    sub = pf[pf["cohort"].isin(top_cohorts)]

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2), sharex=True)
    for cohort, d in sub.groupby("cohort"):
        sns.kdeplot(d["raw"], ax=axes[0], color="#457B9D", lw=0.9, alpha=0.8)
        sns.kdeplot(d["harmonized"], ax=axes[1], color="#E63946", lw=0.9, alpha=0.8)
    axes[0].set_title(f"{gene} — before harmonization")
    axes[1].set_title(f"{gene} — after {SCATTER_ATTEMPTS[1].split('__')[2]}")
    for ax in axes:
        ax.set_xlabel("log2 expression")
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l6_density_before_after")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l6_worst_genes_gallery — are the low-ρ genes broken, or simply flat?
if RUN_SCATTER_SECTION:
    from scipy.stats import spearmanr

    worst = gene_stats.loc[[g for g in gate_genes if g in gene_stats.index]]
    worst = worst.nsmallest(9, "median_rho").index.tolist()
    pf = paired_frame(SCATTER_ATTEMPTS[0], worst)

    fig, axes = plt.subplots(3, 3, figsize=(7.5, 7.5))
    for ax, gene in zip(axes.ravel(), worst):
        d_all = pf[pf["gene"] == gene]
        # Show the cohort where this gene does worst — the diagnostic case.
        scores = {c: spearmanr(d["raw"], d["harmonized"]).statistic
                  for c, d in d_all.groupby("cohort") if len(d) > 5}
        if not scores:
            ax.axis("off")
            continue
        cohort = min(scores, key=lambda c: scores[c])
        d = d_all[d_all["cohort"] == cohort]
        sns.scatterplot(data=d, x="raw", y="harmonized", s=8, alpha=0.6,
                        color="#8D99AE", edgecolor="none", ax=ax)
        ax.set_title(f"{gene} · {cohort[:12]}\nworst ρ = {scores[cohort]:.3f}", fontsize=5)
        ax.set_xlabel("")
        ax.set_ylabel("")
    fig.suptitle("Nine worst genes, each in its worst cohort", y=1.0)
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l6_worst_genes_gallery")
    plt.show()
else:
    print("skipped")

## §9 — Gene–gene correlation structure

Two genes that measure the same thing contribute the same information twice to the panel
mean. This section measures that directly, from the consensus gene × gene correlation
matrices produced by `gene_panel_analysis/`.

**Two flavours, and why both exist.** A *global* correlation between two genes can be
produced entirely by batch structure — both genes higher on one platform than another —
without the genes being co-expressed inside any single cohort. The *within-cohort* flavour
computes the correlation inside each cohort (≥ 20 samples) and averages on the Fisher z
scale (Fisher 1921; Silver & Dunlap 1987). **The within-cohort matrix is the biologically
meaningful one and drives the selection in §10**; the global matrix is shown for contrast.

**The SD matrix** is the SD of Fisher z per gene pair across attempts: a high value means
that apparent co-expression depends on which harmonizer was used, and is not a property of
the biology.

In [ ]:
if not HAS_CONSENSUS:
    print("Consensus matrices not available — §9 and §10 skipped.")
    print("Run gene_panel_analysis/run_gene_corr_parallel.py, then run_gene_corr_concat.py.")
    R_within = pd.DataFrame()
    R_global = pd.DataFrame()
    R_sd = pd.DataFrame()
else:
    cons_genes = [g for g in consensus_within.index if g in ann.index]
    R_within = consensus_within.loc[cons_genes, cons_genes]
    R_global = (consensus_global.loc[cons_genes, cons_genes]
                if not consensus_global.empty else pd.DataFrame())
    R_sd = (consensus_sd.loc[cons_genes, cons_genes]
            if not consensus_sd.empty else pd.DataFrame())

    off = R_within.values[~np.eye(len(R_within), dtype=bool)]
    print(f"Consensus matrix : {R_within.shape[0]} x {R_within.shape[1]}")
    print(f"  |rho| >= 0.9   : {int(np.nansum(np.abs(off) >= 0.9)) // 2} gene pairs")
    print(f"  |rho| >= 0.8   : {int(np.nansum(np.abs(off) >= 0.8)) // 2} gene pairs")
    print(f"  |rho| >= 0.7   : {int(np.nansum(np.abs(off) >= 0.7)) // 2} gene pairs")
    print(f"  mean |rho|     : {np.nanmean(np.abs(off)):.3f}")

In [ ]:
# Fig l7_consensus_clustermap_within — how much redundancy is in the panel.
if HAS_CONSENSUS:
    rc = gene_row_colors(list(R_within.index))
    g = sns.clustermap(
        R_within.fillna(0), cmap="RdBu_r", center=0, vmin=-1, vmax=1, figsize=(12, 12),
        row_colors=rc, col_colors=rc, xticklabels=False, yticklabels=False,
        method="average", cbar_kws={"label": "consensus ρ (within-cohort)"},
    )
    g.figure.suptitle("Consensus gene × gene correlation, within-cohort", y=1.0)
    save_figure(g.figure, "l7_consensus_clustermap_within")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l7_consensus_clustermap_global — the same genes, correlated over all samples.
if HAS_CONSENSUS and not R_global.empty:
    rc = gene_row_colors(list(R_global.index))
    g = sns.clustermap(
        R_global.fillna(0), cmap="RdBu_r", center=0, vmin=-1, vmax=1, figsize=(12, 12),
        row_colors=rc, col_colors=rc, xticklabels=False, yticklabels=False,
        method="average", cbar_kws={"label": "consensus ρ (all samples)"},
    )
    g.figure.suptitle("Consensus gene × gene correlation, all samples pooled "
                      "(contains batch structure)", y=1.0)
    save_figure(g.figure, "l7_consensus_clustermap_global")
    plt.show()

    iu = np.triu_indices(len(R_within), 1)
    both = pd.DataFrame({
        "within_cohort": R_within.values[iu],
        "global": R_global.values[iu],
    }).dropna()
    fig, ax = plt.subplots(figsize=(4.0, 4.0))
    sns.scatterplot(data=both.sample(min(20000, len(both)), random_state=0),
                    x="global", y="within_cohort", s=3, alpha=0.15,
                    color="#457B9D", edgecolor="none", ax=ax)
    ax.plot([-1, 1], [-1, 1], color="k", lw=0.7, ls="--")
    ax.set_title("Pairs far below the diagonal are\nbatch-driven co-expression")
    sns.despine(fig=fig)
    save_figure(fig, "l7_consensus_within_vs_global")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l7_consensus_sd_clustermap — which relationships depend on the harmonizer.
if HAS_CONSENSUS and not R_sd.empty:
    rc = gene_row_colors(list(R_sd.index))
    g = sns.clustermap(
        R_sd.fillna(0), cmap="rocket_r", figsize=(11, 11),
        row_colors=rc, col_colors=rc, xticklabels=False, yticklabels=False,
        method="average", cbar_kws={"label": "SD of Fisher z across attempts"},
    )
    g.figure.suptitle("Stability of each gene–gene relationship", y=1.0)
    save_figure(g.figure, "l7_consensus_sd_clustermap")
    plt.show()
else:
    print("skipped")

In [ ]:
# Fig l7_within_vs_between_set_correlation — is a signature internally coherent?
if HAS_CONSENSUS:
    genes_c = list(R_within.index)
    group_of = {g: set(ann.loc[g, "gene_groups"]) for g in genes_c}
    same_set = np.zeros((len(genes_c), len(genes_c)), dtype=bool)
    for i, gi in enumerate(genes_c):
        for j in range(i + 1, len(genes_c)):
            if group_of[gi] & group_of[genes_c[j]]:
                same_set[i, j] = same_set[j, i] = True

    iu = np.triu_indices(len(genes_c), 1)
    pairs = pd.DataFrame({
        "abs_rho": np.abs(R_within.values[iu]),
        "relation": np.where(same_set[iu], "same signature", "different signatures"),
    }).dropna()

    fig, ax = plt.subplots(figsize=(5.0, 3.2))
    sns.kdeplot(data=pairs, x="abs_rho", hue="relation", common_norm=False,
                fill=True, alpha=0.35,
                palette={"same signature": "#E63946", "different signatures": "#8D99AE"},
                ax=ax)
    ax.set_xlabel("|consensus ρ| (within-cohort)")
    ax.set_title("Genes of one signature should be more correlated\nthan genes of different ones")
    sns.despine(fig=fig)
    save_figure(fig, "l7_within_vs_between_set_correlation")
    plt.show()

    print(pairs.groupby("relation")["abs_rho"].describe()[["count", "mean", "50%"]])
else:
    print("skipped")

In [ ]:
# Fig l7_redundancy_supervenn + Table T5 — redundancy clusters and where they come from.
REDUNDANCY_THRESHOLD = 0.8

if HAS_CONSENSUS:
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components

    A = (np.abs(R_within.fillna(0).values) >= REDUNDANCY_THRESHOLD).astype(int)
    np.fill_diagonal(A, 0)
    n_comp, comp_labels = connected_components(csr_matrix(A), directed=False)
    comp = pd.Series(comp_labels, index=R_within.index, name="component")
    sizes = comp.value_counts()
    multi = sizes[sizes > 1]

    print(f"At |ρ| >= {REDUNDANCY_THRESHOLD}: {len(multi)} redundancy clusters covering "
          f"{int(multi.sum())} genes; {int((sizes == 1).sum())} genes stand alone.")

    try:
        from supervenn import supervenn

        top = multi.head(10).index
        sets = [set(comp.index[comp == c]) for c in top]
        labels = [f"cluster {c} (n={sizes[c]})" for c in top]
        fig, ax = plt.subplots(figsize=(12, 5))
        supervenn(sets, labels, ax=ax, side_plots=False, chunks_ordering="size")
        ax.set_title(f"Largest redundancy clusters at |ρ| >= {REDUNDANCY_THRESHOLD}")
        save_figure(fig, "l7_redundancy_supervenn")
        plt.show()
    except ImportError:
        print("supervenn not installed — cluster figure skipped")

    iu = np.triu_indices(len(R_within), 1)
    T5 = pd.DataFrame({
        "gene_a": np.array(R_within.index)[iu[0]],
        "gene_b": np.array(R_within.index)[iu[1]],
        "consensus_rho": R_within.values[iu],
        "sd_z": R_sd.values[iu] if not R_sd.empty else np.nan,
        "n_attempts": (consensus_n.loc[R_within.index, R_within.index].values[iu]
                       if not consensus_n.empty else np.nan),
    })
    T5 = T5[T5["consensus_rho"].abs() >= 0.7].sort_values("consensus_rho", ascending=False)
    T5.to_csv(FIG_DIR / "T5_gene_pair_redundancy.csv", index=False)
    print(f"saved: {FIG_DIR / 'T5_gene_pair_redundancy.csv'}  ({len(T5)} pairs)")
else:
    print("skipped")

## §10 — Minimal orthogonal panel selection

Four selectors, run on the gate-filtered gene set and compared side by side rather than one
chosen blind. All operate on the within-cohort consensus matrix.

| # | Selector | Reference | Idea |
|---|---|---|---|
| 1 | Hierarchical + medoid | Monti et al. 2003 for the cut height | Cluster on `1 − abs(ρ)`, keep one exemplar (medoid) per cluster |
| 2 | Greedy redundancy pruning | Kuhn 2008, `caret::findCorrelation` | Repeatedly drop the member of the most-correlated pair that is more correlated with everything else |
| 3 | mRMR | Peng, Long & Ding 2005 | Maximise relevance minus mean redundancy to what is already chosen |
| 4 | Affinity propagation | Frey & Dueck 2007 | Exemplar per cluster, without fixing the cluster count |

**Relevance** is the SD of ρ across attempts (§7), not its height — a gene that scores 0.99
everywhere cannot separate harmonizers.

### How a panel is judged

**The reduced panel is not required to reproduce the full panel's ranking of attempts.**
That ranking is dominated by the saturated genes, so agreement with it would reward exactly
what the reduction removes. It is computed and plotted, but as a *descriptive* quantity
only.

| Criterion | Definition | Pass |
|---|---|---|
| Precision | SD of an attempt's panel score under bootstrap resampling of the panel genes | small relative to the between-attempt spread |
| Discriminative SNR | between-attempt variance ÷ mean within-attempt bootstrap variance | maximised |
| External agreement | Spearman of the panel's attempt ranking against *independent* metrics (Group M `xb_*`, Group A `r2_RNA_BATCH`, Group B kBET/iLISI) | positive and stable |
| Structural | coverage complete, QC passed, max within-panel abs(ρ) below threshold, at least one gene per represented cell type | hard gate |

In [ ]:
if HAS_CONSENSUS:
    sel_genes = [g for g in working_genes if g in R_within.index]
    R = R_within.loc[sel_genes, sel_genes].fillna(0.0)
    absR = R.abs().values.copy()
    np.fill_diagonal(absR, 0.0)
    relevance = gene_stats.loc[sel_genes, "sd_rho"].fillna(0.0).values

    print(f"Selection pool: {len(sel_genes)} genes "
          f"(coverage + QC + saturation gates applied)")
    print(f"  max off-diagonal |ρ| : {absR.max():.3f}")
    print(f"  mean relevance (SD ρ): {relevance.mean():.3f}")
else:
    sel_genes, R, absR, relevance = [], pd.DataFrame(), np.zeros((0, 0)), np.zeros(0)
    print("skipped — no consensus matrix")

In [ ]:
def select_hierarchical(absR, genes, height):
    """
    Cluster on 1 - abs(rho) and keep each cluster's medoid.

    Returns (chosen genes, cluster label per input gene).
    """
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    D = 1.0 - absR
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="average")
    labels = fcluster(Z, t=height, criterion="distance")
    chosen = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        medoid = idx[np.argmin(D[np.ix_(idx, idx)].mean(axis=1))]
        chosen.append(genes[medoid])
    return sorted(chosen), labels


def select_find_correlation(absR, genes, cutoff):
    """
    caret::findCorrelation (Kuhn 2008).

    Repeatedly take the most-correlated surviving pair and drop whichever member is more
    correlated with everything else, until no pair exceeds the cutoff.
    """
    A = absR.copy()
    alive = np.ones(len(genes), dtype=bool)
    while True:
        masked = np.where(np.outer(alive, alive), A, -1.0)
        i, j = np.unravel_index(np.argmax(masked), masked.shape)
        if masked[i, j] < cutoff:
            break
        alive[i if A[i][alive].mean() > A[j][alive].mean() else j] = False
    return sorted(np.array(genes)[alive].tolist())


def select_mrmr(absR, genes, relevance, k):
    """Greedy maximum relevance, minimum redundancy (Peng, Long & Ding 2005)."""
    chosen = [int(np.argmax(relevance))]
    while len(chosen) < min(k, len(genes)):
        score = relevance - absR[:, chosen].mean(axis=1)
        score[chosen] = -np.inf
        chosen.append(int(np.argmax(score)))
    return sorted(np.array(genes)[chosen].tolist())


def select_affinity(absR, genes):
    """Affinity propagation exemplars (Frey & Dueck 2007) — no cluster count to choose."""
    from sklearn.cluster import AffinityPropagation

    S = absR.copy()
    np.fill_diagonal(S, 1.0)
    ap = AffinityPropagation(affinity="precomputed", random_state=0, damping=0.9,
                             max_iter=500).fit(S)
    return sorted(np.array(genes)[ap.cluster_centers_indices_].tolist())


print("Selectors defined.")

In [ ]:
# Fig l8_dendrogram_cut_sweep — the cut height trades panel size against redundancy.
CUT_HEIGHT = 0.4          # on 1 - abs(rho); 0.4 means cluster members share abs(rho) >= 0.6
FIND_CORR_CUTOFF = 0.75
MRMR_K = 30

if HAS_CONSENSUS and len(sel_genes) > 3:
    heights = np.round(np.arange(0.05, 0.85, 0.025), 3)
    rows = []
    for h in heights:
        chosen, _ = select_hierarchical(absR, sel_genes, h)
        idx = [sel_genes.index(g) for g in chosen]
        sub = absR[np.ix_(idx, idx)]
        rows.append({"height": h, "n_genes": len(chosen),
                     "max_abs_rho": float(sub.max()) if len(idx) > 1 else 0.0,
                     "mean_relevance": float(relevance[idx].mean())})
    sweep10 = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.0))
    sns.lineplot(data=sweep10, x="height", y="n_genes", color="#1D3557", lw=1.4, ax=axes[0])
    axes[0].axvline(CUT_HEIGHT, color="#E63946", ls="--", lw=0.9)
    axes[0].set_xlabel("cut height on 1 − |ρ|")
    axes[0].set_ylabel("genes retained")
    axes[0].set_title("Panel size")

    sns.lineplot(data=sweep10, x="height", y="max_abs_rho", color="#E63946", lw=1.4,
                 ax=axes[1], label="max |ρ| within panel")
    sns.lineplot(data=sweep10, x="height", y="mean_relevance", color="#457B9D", lw=1.4,
                 ax=axes[1], label="mean relevance")
    axes[1].axvline(CUT_HEIGHT, color="k", ls="--", lw=0.7)
    axes[1].set_xlabel("cut height on 1 − |ρ|")
    axes[1].set_title("Orthogonality and informativeness")
    axes[1].legend(frameon=False, fontsize=6)
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l8_dendrogram_cut_sweep")
    plt.show()
else:
    print("skipped")

In [ ]:
# Run all four selectors.
PANELS = {}
if HAS_CONSENSUS and len(sel_genes) > 3:
    PANELS["hierarchical"] = select_hierarchical(absR, sel_genes, CUT_HEIGHT)[0]
    PANELS["findCorrelation"] = select_find_correlation(absR, sel_genes, FIND_CORR_CUTOFF)
    PANELS["mRMR"] = select_mrmr(absR, sel_genes, relevance, MRMR_K)
    try:
        PANELS["affinity"] = select_affinity(absR, sel_genes)
    except Exception as exc:
        print(f"affinity propagation failed: {exc}")
    PANELS["gate_filtered"] = sel_genes
    PANELS["full_panel"] = list(rho_wide.index)

    for name, panel in PANELS.items():
        idx = [sel_genes.index(g) for g in panel if g in sel_genes]
        mx = float(absR[np.ix_(idx, idx)].max()) if len(idx) > 1 else np.nan
        print(f"{name:<16} {len(panel):>4} genes   max within-panel |ρ| = {mx:.3f}")
else:
    print("skipped")

In [ ]:
# Fig l8_selector_overlap_supervenn — do the four selectors agree?
if PANELS:
    try:
        from supervenn import supervenn

        names = [n for n in ["hierarchical", "findCorrelation", "mRMR", "affinity"]
                 if n in PANELS]
        fig, ax = plt.subplots(figsize=(11, 4))
        supervenn([set(PANELS[n]) for n in names], names, ax=ax,
                  side_plots=True, chunks_ordering="size")
        ax.set_title("Genes chosen by each selector")
        save_figure(fig, "l8_selector_overlap_supervenn")
        plt.show()
    except ImportError:
        print("supervenn not installed")
else:
    print("skipped")

In [ ]:
# Fig l8_selected_panel_clustermap — the chosen panel should be near block-diagonal.
PRIMARY_SELECTOR = "hierarchical"

if PANELS and PRIMARY_SELECTOR in PANELS:
    panel = [g for g in PANELS[PRIMARY_SELECTOR] if g in R.index]
    rc = gene_row_colors(panel)
    g = sns.clustermap(
        R.loc[panel, panel], cmap="RdBu_r", center=0, vmin=-1, vmax=1, figsize=(9, 9),
        row_colors=rc, col_colors=rc, xticklabels=True, yticklabels=True,
        method="average", cbar_kws={"label": "consensus ρ"},
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=4.5, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=4.5)
    g.figure.suptitle(f"Selected panel ({PRIMARY_SELECTOR}, {len(panel)} genes)", y=1.0)
    save_figure(g.figure, "l8_selected_panel_clustermap")
    plt.show()
else:
    print("skipped")

In [ ]:
# The D7 criteria, evaluated per candidate panel.
N_BOOTSTRAP = 500
EXTERNAL_METRICS = [m for m in
                    ["xb_rank_agree_ratio", "r2_RNA_BATCH",
                     "kbet_acceptance_rate_RNA_BATCH", "ilisi_RNA_BATCH",
                     "pv_lobo3_f1_macro_mean"]
                    if m in df_metrics.columns]
print(f"Independent metrics available for the external check: {EXTERNAL_METRICS}")


def panel_scores(panel):
    """Mean ρ per attempt over one panel — the reduced mk_rho_mean_all_genes."""
    genes = [g for g in panel if g in rho_wide.index]
    return rho_wide.loc[genes].mean(axis=0, skipna=True).dropna()


def evaluate_panel(panel, rng):
    """Bootstrap precision, discriminative SNR, and agreement with independent metrics."""
    from scipy.stats import spearmanr

    genes = [g for g in panel if g in rho_wide.index]
    scores = panel_scores(genes)
    sub = rho_wide.loc[genes, scores.index].values

    boot = np.empty((N_BOOTSTRAP, sub.shape[1]))
    for b in range(N_BOOTSTRAP):
        pick = rng.integers(0, sub.shape[0], sub.shape[0])
        boot[b] = np.nanmean(sub[pick], axis=0)
    within_var = float(np.nanmean(np.nanvar(boot, axis=0)))
    between_var = float(np.nanvar(scores.values))

    out = {
        "n_genes": len(genes),
        "score_mean": float(scores.mean()),
        "score_sd_between_attempts": float(np.sqrt(between_var)),
        "bootstrap_sd": float(np.sqrt(within_var)),
        "discriminative_snr": (between_var / within_var) if within_var > 0 else np.nan,
    }
    for m in EXTERNAL_METRICS:
        joined = pd.concat([scores.rename("panel"), df_metrics[m]], axis=1).dropna()
        out[f"agree_{m}"] = (float(spearmanr(joined["panel"], joined[m]).statistic)
                             if len(joined) > 10 else np.nan)

    full = panel_scores(list(rho_wide.index))
    joined = pd.concat([scores.rename("panel"), full.rename("full")], axis=1).dropna()
    # Descriptive only — see the section header for why this is not an acceptance criterion.
    out["descriptive_agree_full_panel"] = (
        float(spearmanr(joined["panel"], joined["full"]).statistic)
        if len(joined) > 10 else np.nan
    )
    return out


if PANELS:
    rng = np.random.default_rng(260827)
    T6 = pd.DataFrame(
        [{"selector": name, **evaluate_panel(panel, rng)} for name, panel in PANELS.items()]
    ).sort_values("discriminative_snr", ascending=False)
    T6.to_csv(FIG_DIR / "T6_candidate_panels.csv", index=False)
    print(f"saved: {FIG_DIR / 'T6_candidate_panels.csv'}")
    print(T6[["selector", "n_genes", "bootstrap_sd", "score_sd_between_attempts",
              "discriminative_snr", "descriptive_agree_full_panel"]].to_string(index=False))
else:
    T6 = pd.DataFrame()
    print("skipped")

In [ ]:
# Fig l8_panel_size_vs_signal — THE decision plot: panel size against the D7 criteria.
if PANELS and not T6.empty:
    rng = np.random.default_rng(260827)
    sizes = [k for k in [5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 75, 100] if k <= len(sel_genes)]
    rows = []
    for k in sizes:
        rec = evaluate_panel(select_mrmr(absR, sel_genes, relevance, k), rng)
        rec["selector"], rec["k"] = "mRMR", k
        rows.append(rec)
    curve = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    sns.lineplot(data=curve, x="k", y="bootstrap_sd", marker="o", color="#457B9D", ax=axes[0])
    axes[0].set_title("Precision\n(bootstrap SD of the panel score)")
    axes[0].set_xlabel("panel size")

    sns.lineplot(data=curve, x="k", y="discriminative_snr", marker="o", color="#E63946",
                 ax=axes[1])
    knee = curve.loc[curve["discriminative_snr"].idxmax(), "k"]
    axes[1].axvline(knee, color="k", ls="--", lw=0.8)
    axes[1].set_title(f"Discriminative SNR\n(best at k = {int(knee)})")
    axes[1].set_xlabel("panel size")

    for m in EXTERNAL_METRICS:
        sns.lineplot(data=curve, x="k", y=f"agree_{m}", marker="o", lw=1.1, label=m, ax=axes[2])
    sns.lineplot(data=curve, x="k", y="descriptive_agree_full_panel", color="grey",
                 ls="--", lw=1.0, label="full panel (descriptive,\nnot a criterion)", ax=axes[2])
    axes[2].axhline(0, color="k", lw=0.6)
    axes[2].set_title("Agreement with independent metrics")
    axes[2].set_xlabel("panel size")
    axes[2].set_ylabel("Spearman ρ of attempt rankings")
    axes[2].legend(frameon=False, fontsize=4.5, loc="best")
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l8_panel_size_vs_signal")
    plt.show()

    curve.to_csv(FIG_DIR / "T6b_panel_size_curve.csv", index=False)
    print(f"saved: {FIG_DIR / 'T6b_panel_size_curve.csv'}")
else:
    curve = pd.DataFrame()
    print("skipped")

In [ ]:
# Table T7 — the recommended panel, one row per gene with the reason for inclusion.
if PANELS and PRIMARY_SELECTOR in PANELS:
    _, labels = select_hierarchical(absR, sel_genes, CUT_HEIGHT)
    cluster_of = dict(zip(sel_genes, labels))
    members = {}
    for gene, c in cluster_of.items():
        members.setdefault(c, []).append(gene)

    T7 = pd.DataFrame({"gene": PANELS[PRIMARY_SELECTOR]})
    T7["cluster"] = T7["gene"].map(cluster_of)
    T7["cluster_size"] = T7["cluster"].map(lambda c: len(members.get(c, [])))
    T7["represents"] = T7["cluster"].map(lambda c: ";".join(sorted(members.get(c, []))))
    T7 = T7.join(gene_stats[["median_rho", "sd_rho"]], on="gene")
    T7 = T7.join(coverage[["frac_raw_pairs"]], on="gene")
    T7["primary_cell_type"] = T7["gene"].map(ann["primary_cell_type"])
    T7["gene_groups"] = T7["gene"].map(lambda g_: ";".join(ann.loc[g_, "gene_groups"]))
    if HAS_QC:
        T7 = T7.join(qc_gene[["zero_frac", "frac_lt_1", "icc_cohort", "genorm_m"]], on="gene")
    T7["reason"] = np.where(
        T7["cluster_size"] > 1,
        "medoid of a " + T7["cluster_size"].astype(str) + "-gene redundancy cluster",
        "orthogonal — no correlated partner above the threshold",
    )
    T7 = T7.sort_values("sd_rho", ascending=False)
    T7.to_csv(FIG_DIR / "T7_recommended_panel.csv", index=False)
    print(f"saved: {FIG_DIR / 'T7_recommended_panel.csv'}  ({len(T7)} genes)")
    print(T7[["gene", "median_rho", "sd_rho", "cluster_size", "primary_cell_type"]]
          .head(20).to_string(index=False))

    covered_types = set(T7["primary_cell_type"])
    missing_types = set(ann.loc[working_genes, "primary_cell_type"]) - covered_types
    print(f"\nCell types represented: {len(covered_types)}")
    if missing_types:
        print(f"Cell types lost by the reduction: {sorted(missing_types)}")
else:
    T7 = pd.DataFrame()
    print("skipped")

## §11 — Variance, noise and coherence

Everything above judges a gene by how its *correlation* behaves. This section judges it by
how its *expression* behaves — whether it is measured reliably enough to deserve a place in
a small panel.

| Figure | Method | Reference | How to read it |
|---|---|---|---|
| `l9_mean_variance_trend` | LOWESS trend of per-gene SD against mean log-expression | Law et al. 2014 (*voom*), Genome Biol 15:R29 | Variance depends on the mean by construction. Genes far **above** the trend are noisier than their expression level explains |
| `l9_cv2_vs_mean` | CV² against mean on log axes with a fitted trend | Brennecke et al. 2013, Nat Methods 10:1093 | Points **above** the trend are genuinely variable; points below are flat and carry little information |
| `l9_icc_distribution` | ICC(1) = between-cohort variance ÷ total | Shrout & Fleiss 1979; McGraw & Wong 1996 | **High ICC is bad here**: the variance is mostly *between* cohorts, so the within-cohort ranks that Group L measures rest on very little real spread |
| `l9_genorm_stability` | geNorm **M**: mean over other genes of the SD of the pairwise log-ratio, computed within cohorts and averaged | Vandesompele et al. 2002, Genome Biol 3:research0034 | **Low M = more stable.** A marker below the housekeeping median is exceptionally well-behaved |
| `l9_signature_coherence` | Cronbach's α = `k·r̄ / (1 + (k−1)·r̄)` from the mean inter-gene correlation `r̄`; and PC1 variance explained | Cronbach 1951; Langfelder & Horvath 2008 | α ≥ 0.7 is the conventional adequacy threshold. A signature below it is not measuring one thing |
| `l9_kme_by_gene` | PC1 loading of each gene inside its own signature — a kME proxy computed from the consensus matrix | Langfelder & Horvath 2008, BMC Bioinformatics 9:559 | Genes near zero sit at the edge of their signature and are the first to drop |
| `l9_noise_vs_rho` | Excess noise against median ρ | — | Answers whether the low-ρ genes are broken or merely noisy |

Shrinking a per-gene variance towards the fitted trend (Love, Huber & Anders 2014, DESeq2)
is the reason the *residual* from the trend is used for ranking rather than the raw variance.

In [ ]:
# Fig l9_mean_variance_trend — voom-style trend and Brennecke-style overdispersion.
if HAS_QC:
    import statsmodels.api as sm

    nv = qc_gene.dropna(subset=["mean_expr", "sd_expr"]).copy()
    nv = nv[nv["mean_expr"] > 0]
    nv["cv2"] = (nv["sd_expr"] / nv["mean_expr"]) ** 2
    trend = sm.nonparametric.lowess(nv["sd_expr"], nv["mean_expr"], frac=0.4,
                                    return_sorted=False)
    nv["excess_noise"] = nv["sd_expr"] - trend

    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.6))
    sns.scatterplot(data=nv.reset_index(), x="mean_expr", y="sd_expr",
                    hue="primary_cell_type", palette=cell_type_pal, s=14, alpha=0.8,
                    edgecolor="none", legend=False, ax=axes[0])
    sns.regplot(data=nv, x="mean_expr", y="sd_expr", lowess=True, scatter=False,
                line_kws={"color": "k", "lw": 1.2}, ax=axes[0])
    hk = nv[nv["is_housekeeping"]]
    axes[0].scatter(hk["mean_expr"], hk["sd_expr"], marker="x", s=18, color="k",
                    label="housekeeping")
    axes[0].legend(frameon=False, fontsize=6)
    axes[0].set_xlabel("mean log2 expression")
    axes[0].set_ylabel("SD")
    axes[0].set_title("Mean–variance trend (voom-style)")

    sns.scatterplot(data=nv.reset_index(), x="mean_expr", y="cv2",
                    hue="primary_cell_type", palette=cell_type_pal, s=14, alpha=0.8,
                    edgecolor="none", legend=False, ax=axes[1])
    sns.regplot(data=nv, x="mean_expr", y="cv2", lowess=True, scatter=False,
                line_kws={"color": "k", "lw": 1.2}, ax=axes[1])
    axes[1].set_yscale("log")
    axes[1].set_xlabel("mean log2 expression")
    axes[1].set_ylabel("CV²")
    axes[1].set_title("Overdispersion (Brennecke-style)")
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l9_mean_variance_trend")
    plt.show()
else:
    nv = pd.DataFrame()
    print("skipped — no QC table")

In [ ]:
# Fig l9_icc_distribution — high ICC marks a batch-driven, and therefore poor, marker.
if HAS_QC:
    icc = qc_gene.dropna(subset=["icc_cohort"]).sort_values("icc_cohort")
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
    sns.histplot(icc["icc_cohort"], bins=40, color="#457B9D", ax=axes[0])
    axes[0].set_xlabel("ICC(1): between-cohort variance / total")
    axes[0].set_title("Cohort structure per gene")

    show = pd.concat([icc.head(15), icc.tail(15)]).reset_index()
    sns.barplot(data=show, y="gene", x="icc_cohort", hue="primary_cell_type",
                palette=cell_type_pal, dodge=False, ax=axes[1])
    axes[1].set_title("15 lowest and 15 highest ICC")
    axes[1].tick_params(labelsize=4.5)
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5)
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l9_icc_distribution")
    plt.show()
else:
    print("skipped — no QC table")

In [ ]:
# Fig l9_genorm_stability — marker stability against the housekeeping controls.
if HAS_QC:
    gm = qc_gene.dropna(subset=["genorm_m"]).copy()
    gm["band"] = np.where(gm["is_housekeeping"], "housekeeping", "marker")
    hk_median = gm.loc[gm["is_housekeeping"], "genorm_m"].median()

    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.4))
    sns.boxplot(data=gm, x="band", y="genorm_m",
                palette={"housekeeping": "#6C757D", "marker": "#E63946"},
                fliersize=0.6, linewidth=0.7, ax=axes[0])
    axes[0].set_title("geNorm M — lower is more stable")
    axes[0].set_xlabel("")

    best = gm[~gm["is_housekeeping"]].nsmallest(20, "genorm_m").reset_index()
    sns.barplot(data=best, y="gene", x="genorm_m", hue="primary_cell_type",
                palette=cell_type_pal, dodge=False, ax=axes[1])
    axes[1].axvline(hk_median, color="k", ls="--", lw=0.9, label="housekeeping median")
    axes[1].set_title("20 most stable marker genes")
    axes[1].tick_params(labelsize=5)
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=5)
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l9_genorm_stability")
    plt.show()

    print(f"Markers more stable than the housekeeping median: "
          f"{int((gm.loc[~gm['is_housekeeping'], 'genorm_m'] < hk_median).sum())}")
else:
    print("skipped — no QC table")

In [ ]:
# Fig l9_signature_coherence — Cronbach's alpha and PC1 variance explained per signature.
if HAS_CONSENSUS:
    rows = []
    for grp in GENE_GROUPS:
        genes_in = [g for g in ann_raw.loc[ann_raw["gene_group"] == grp, "gene"].unique()
                    if g in R_within.index]
        k = len(genes_in)
        if k < 3:
            continue
        sub = np.nan_to_num(R_within.loc[genes_in, genes_in].values, nan=0.0)
        rbar = float(sub[np.triu_indices(k, 1)].mean())
        denom = 1 + (k - 1) * rbar
        alpha = (k * rbar / denom) if denom != 0 else np.nan
        eig = np.linalg.eigvalsh(sub)
        rows.append({"gene_group": grp, "n_genes": k, "mean_inter_rho": rbar,
                     "cronbach_alpha": alpha, "pc1_var_explained": float(eig[-1] / k)})
    coherence = pd.DataFrame(rows).sort_values("cronbach_alpha", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
    sns.barplot(data=coherence, x="gene_group", y="cronbach_alpha",
                palette=gene_group_pal, ax=axes[0])
    axes[0].axhline(0.7, color="k", ls="--", lw=0.9)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90, fontsize=4.5)
    axes[0].set_xlabel("")
    axes[0].set_title("Cronbach's α per signature (0.7 = conventional adequacy)")

    sns.scatterplot(data=coherence, x="cronbach_alpha", y="pc1_var_explained",
                    size="n_genes", hue="gene_group", palette=gene_group_pal,
                    sizes=(10, 120), legend=False, ax=axes[1])
    axes[1].set_title("Coherence: α vs PC1 variance explained")
    sns.despine(fig=fig)
    fig.tight_layout()
    save_figure(fig, "l9_signature_coherence")
    plt.show()

    print(f"Signatures with α >= 0.7: {(coherence['cronbach_alpha'] >= 0.7).sum()} "
          f"of {len(coherence)}")
else:
    coherence = pd.DataFrame()
    print("skipped — no consensus matrix")

In [ ]:
# Fig l9_kme_by_gene — each gene's PC1 loading inside its own signature.
if HAS_CONSENSUS and not coherence.empty:
    kme_rows = []
    for grp in coherence["gene_group"]:
        genes_in = [g for g in ann_raw.loc[ann_raw["gene_group"] == grp, "gene"].unique()
                    if g in R_within.index]
        sub = np.nan_to_num(R_within.loc[genes_in, genes_in].values, nan=0.0)
        _, vecs = np.linalg.eigh(sub)
        pc1 = vecs[:, -1]
        if pc1.sum() < 0:      # the sign of an eigenvector is arbitrary
            pc1 = -pc1
        kme_rows.extend({"gene": g_, "gene_group": grp, "kme": float(v)}
                        for g_, v in zip(genes_in, pc1))
    kme = pd.DataFrame(kme_rows)

    fig, ax = plt.subplots(figsize=(9.5, 3.4))
    sns.boxplot(data=kme, x="gene_group", y="kme", order=coherence["gene_group"].tolist(),
                palette=gene_group_pal, fliersize=0.6, linewidth=0.5, ax=ax)
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=4.5)
    ax.set_xlabel("")
    ax.set_ylabel("PC1 loading within the signature (kME proxy)")
    ax.set_title("Genes near zero sit at the edge of their signature")
    sns.despine(fig=fig)
    save_figure(fig, "l9_kme_by_gene")
    plt.show()
else:
    kme = pd.DataFrame()
    print("skipped")

In [ ]:
# Fig l9_noise_vs_rho — are the low-ρ genes broken, or simply noisy?
if HAS_QC and not nv.empty:
    joined = gene_stats.join(nv[["excess_noise", "icc_cohort", "genorm_m"]], how="inner")
    joined["survives_gates"] = joined.index.isin(working_genes)

    fig, ax = plt.subplots(figsize=(5.2, 4.0))
    sns.scatterplot(data=joined.reset_index(), x="excess_noise", y="median_rho",
                    size="sd_rho", hue="survives_gates",
                    palette={True: "#E63946", False: "#8D99AE"},
                    sizes=(8, 90), alpha=0.8, edgecolor="none", ax=ax)
    ax.axhline(SATURATION_MEDIAN_MAX, color="k", ls="--", lw=0.8)
    ax.set_xlabel("excess noise (SD above the mean–variance trend)")
    ax.set_ylabel("median ρ across attempts")
    ax.set_title("Noise vs correlation preservation")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=6)
    sns.despine(fig=fig)
    save_figure(fig, "l9_noise_vs_rho")
    plt.show()

    T8 = joined.reset_index()
    T8.to_csv(FIG_DIR / "T8_gene_noise_coherence.csv", index=False)
    print(f"saved: {FIG_DIR / 'T8_gene_noise_coherence.csv'}")
else:
    T8 = pd.DataFrame()
    print("skipped — no QC table")

## §12 — Recommendation and export

`T9_supplementary_marker_panel_selection.csv` is the full audit trail: one row per gene,
every statistic computed above and the outcome of every gate, ready to become a
supplementary table.

**Acting on the result is deliberately a separate step.** `run_metrics_job.py` currently
accepts only `--panel-groups` (gene *group* names); using an arbitrary gene list would need
a new `--panel-file` argument and a Group L re-run. Nothing in this notebook changes a
published metric.

In [ ]:
summary = pd.DataFrame([
    {"step": "annotated genes", "n": int(ann.index.nunique())},
    {"step": "resolved in at least one matrix", "n": int(rho_wide.shape[0])},
    {"step": f"coverage gate ({COVERAGE_SCOPE})", "n": len(covered_genes)},
    {"step": "expression QC gate", "n": len(gate_genes) if HAS_QC else None},
    {"step": f"saturation gate (median rho <= {SATURATION_MEDIAN_MAX})", "n": len(working_genes)},
    {"step": f"orthogonality ({PRIMARY_SELECTOR})",
     "n": len(PANELS.get(PRIMARY_SELECTOR, [])) if PANELS else None},
])
print(summary.to_string(index=False))

if not T6.empty:
    best = T6.sort_values("discriminative_snr", ascending=False).iloc[0]
    print()
    print(f"Best discriminative SNR: {best['selector']} with {int(best['n_genes'])} genes "
          f"(SNR {best['discriminative_snr']:.1f}, bootstrap SD {best['bootstrap_sd']:.4f})")

In [ ]:
# Table T9 — the complete audit trail, one row per gene.
T9 = gene_stats.copy()
T9 = T9.join(coverage[["frac_ref_strategy"]])
T9["saturated"] = T9.index.map(sat["saturated"])
T9["passes_coverage"] = T9.index.isin(covered_genes)
T9["passes_qc"] = T9.index.isin(gate_genes) if HAS_QC else np.nan
T9["in_working_set"] = T9.index.isin(working_genes)
for name, panel in (PANELS or {}).items():
    T9[f"selected_{name}"] = T9.index.isin(panel)
if HAS_QC:
    T9 = T9.join(qc_gene[["zero_frac", "frac_lt_1", "detection_frac", "mean_expr",
                          "sd_expr", "cv", "icc_cohort", "genorm_m"]])
if not kme.empty:
    T9 = T9.join(kme.groupby("gene")["kme"].mean().rename("kme_mean"))
T9["gene_groups"] = T9.index.map(lambda g_: ";".join(ann.loc[g_, "gene_groups"]))
T9["cell_types"] = T9.index.map(lambda g_: ";".join(ann.loc[g_, "cell_types"]))
T9["sources"] = T9.index.map(lambda g_: ";".join(ann.loc[g_, "sources"]))

T9 = T9.reset_index().sort_values("sd_rho", ascending=False)
T9.to_csv(FIG_DIR / "T9_supplementary_marker_panel_selection.csv", index=False)
print(f"saved: {FIG_DIR / 'T9_supplementary_marker_panel_selection.csv'}  "
      f"({len(T9)} genes x {len(T9.columns)} columns)")
T9.head(15)

In [ ]:
# How the selected panel would be fed back into the compute pipeline.
if PANELS and PRIMARY_SELECTOR in PANELS:
    panel_path = FIG_DIR / "recommended_marker_panel.txt"
    panel_path.write_text("\n".join(sorted(PANELS[PRIMARY_SELECTOR])) + "\n")
    print(f"saved: {panel_path}  ({len(PANELS[PRIMARY_SELECTOR])} genes)")
    print()
    print("To score the benchmark with this panel, run_metrics_job.py needs a new")
    print("--panel-file argument (it currently accepts only --panel-groups). That change,")
    print("and the Group L re-run behind it, are deliberately out of scope here:")
    print()
    print("  python run_metrics_job.py --strat ... --imp ... --method ... \\")
    print(f"      --groups L --force-groups L --panel-file {panel_path.name}")
else:
    print("skipped")